# Отладка предсказания по шагам

У класса `Model` в `utils.py` нет метода `predict`. Единственный вызов XGBoost — `booster.predict(..., output_margin=True)` внутри `Model._margin`. Всё, что показывает проект, — арифметика над этим одним числом на дрон: `risk`, накопленная опасность, выживаемость, условная выживаемость, интенсивность, остаточный ресурс и все кривые дашборда.

Ноутбук повторяет эту арифметику вручную для одного дрона, по одному именованному промежуточному результату за раз, и сверяет каждый шаг с кодом библиотеки. Он нужен для отладки: когда число или кривая выглядят неверно, найдите шаг, на котором значение, посчитанное вручную, расходится со значением библиотеки.

**План**

| Разделы | Что прослеживается | Проверяемый код |
| --- | --- | --- |
| 1 | настройка, отладочные функции, состояние модели, строка дрона | `Model.load` |
| 2 | сырая строка → таблица признаков → `DMatrix` → margin; разложение margin по деревьям и по признакам | `_features_frame`, `Model._margin`, `Model._predict_margin`, `Model.explain` |
| 3–7 | margin → risk → накопленная опасность → выживаемость → условная выживаемость → интенсивность, для боевой модели AFT | `Model.risk`, `_cum_hazard_at`, `_survival_at`, `survival`, `conditional_survival`, `hazard` |
| 8 | запись наблюдения, с которой работают метрики | `_surv` |
| 9 | та же строка на модели Cox | базовая линия Бреслоу, интенсивность, сглаженная ядром, связь хрупкости |
| 10 | вся тестовая таблица: шейпы пакетного расчёта, сохранённые предсказания, метрики | `evaluate`, `artifacts/predictions.parquet`, `artifacts/metrics.parquet` |
| 11–15 | какие массивы рисует каждый график дашборда | `dashboard.common`, `dashboard.figures` |
| 16 | индекс всех промежуточных результатов, сохранённых по ходу | `TRACE` |

**Обозначения**

- Шейпы: `n` строк (дронов), `m` моментов времени, `p` признаков, `T` деревьев, участвующих в предсказании, `k` различных моментов отказа, `g` точек сетки графика. Кривые по дронам всегда имеют шейп `(n, m)`: строка на дрон, столбец на момент времени. До раздела 10 `n = 1`.
- `peek(name, obj)` печатает тип, шейп, dtype, число элементов, объём памяти и превью и сохраняет объект в `TRACE[name]`.
- `same(name, got, expected)` печатает шейп и наибольшую абсолютную разницу и падает, если шейпы или значения не совпали.
- Каждый промежуточный результат — ещё и переменная верхнего уровня с уникальным именем, так что любой из них можно рассмотреть в новой ячейке после прогона.

Проход идёт по **боевым** моделям (`models/production_*.json`): они переобучены на train + validation в разделе 8 `drone_survival.ipynb`, оценивают новых дронов и рисуют карточку дрона. `forecast_name` в `artifacts/meta.json` называет ту, по которой строится прогноз для обслуживания (раздел 14). Оценочные модели (`models/best_*.json`) нужны только там, где пересчитываются опубликованные тестовые метрики и кривая интенсивности флота (разделы 10 и 15). Основной проход (разделы 2–8) написан для AFT; раздел 9 повторяет его для Cox, у которого другие ветки в `_cum_hazard_at`, `hazard` и `conditional_survival`. Чтобы отладить другой дрон, поменяйте `DRONE_ID` в разделе 1.4 и перезапустите ноутбук.

`models/`, `artifacts/` и `data/` только читаются.

## 1. Настройка

### 1.1 Импорты и параметры отображения

Логгер `utils` выводится в ячейку на уровне `WARNING`, поэтому предупреждения библиотеки (неизвестная категория в разделе 2.1, плоская базовая линия Cox в разделе 9.5) появляются рядом с вызовом, который их вызвал.

In [1]:
import dataclasses
import json
import logging
import sys
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xgboost as xgb
from IPython.display import display
from scipy.special import expit
from scipy.stats import norm

from dashboard.common import Settings, drone_hazard_grid, hazard_table
from dashboard.data import DashboardData
from dashboard.figures import conditional_curves, drone_hazard, drone_survival
from dashboard.figures import hazard as fleet_hazard
from dashboard.labels import HAZARD_SCALE, MODEL_LABELS
from utils import (
    SCALAR_METRICS,
    Model,
    _aft_noise,
    _epanechnikov,
    _features_frame,
    _frailty_link,
    _surv,
    encode_unknown_categories,
    evaluate,
    hazard_bandwidth,
    observed_hazard,
    population_hazard,
    remaining_life,
)

log = logging.getLogger("utils")
log.setLevel(logging.WARNING)
log.propagate = False
if not log.handlers:
    log.addHandler(logging.StreamHandler(sys.stdout))

pd.set_option("display.float_format", lambda v: f"{v:.6g}")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

### 1.2 Функции для просмотра массивов

В каждом разделе ниже используются три функции:

- `peek(name, obj)` печатает, что это за объект (тип, шейп, dtype, число элементов, байты), диапазон значений числового массива и короткое превью. Она же сохраняет объект в словаре `TRACE` под именем `name`. В именах стоит номер раздела, например `TRACE["4.z"]`.
- `same(name, got, expected)` — сравнение, на котором держится каждая сверка. Шейпы должны быть равны, без неявного broadcasting, а значения должны совпадать с `rtol=1e-10`, если для данных `float32` не передан более мягкий допуск.
- `trace_index()` возвращает таблицу всего, что уже сохранено; раздел 16 печатает её целиком.

In [2]:
TRACE: dict[str, object] = {}
_SCALARS = (str, int, float, bool, type(None), np.generic)
_CONTAINERS = (dict, list, tuple, set)


def facts(obj: object) -> dict[str, object]:
    """Тип, шейп, dtype, число элементов и память ``obj``; ``None`` там, где величина не определена."""
    row: dict[str, object] = dict.fromkeys(("type", "shape", "dtype", "size", "bytes"))
    row["type"] = type(obj).__name__
    if isinstance(obj, np.ndarray):
        row.update(shape=obj.shape, dtype=str(obj.dtype), size=obj.size, bytes=obj.nbytes)
    elif isinstance(obj, pd.DataFrame):
        counts = obj.dtypes.astype(str).value_counts()
        row.update(
            shape=obj.shape,
            dtype=", ".join(f"{n} x {name}" for name, n in counts.items()),
            size=obj.size,
            bytes=int(obj.memory_usage(deep=True).sum()),
        )
    elif isinstance(obj, pd.Series):
        row.update(
            shape=obj.shape,
            dtype=str(obj.dtype),
            size=obj.size,
            bytes=int(obj.memory_usage(deep=True)),
        )
    elif isinstance(obj, xgb.DMatrix):
        row.update(shape=(obj.num_row(), obj.num_col()), size=obj.num_row() * obj.num_col())
    elif isinstance(obj, (*_CONTAINERS, str)):
        row.update(size=len(obj))
    return row


def _facts_line(obj: object) -> str:
    """``facts`` одной строкой, без неопределённых полей."""
    row = facts(obj)
    keys = ("shape", "dtype", "size", "bytes")
    return " ".join([str(row["type"]), *(f"{key}={row[key]}" for key in keys if row[key] is not None)])


def _brief(obj: object) -> str:
    """Само значение, если это короткий встроенный объект, иначе его тип и шейп."""
    if isinstance(obj, (*_SCALARS, *_CONTAINERS)) and len(repr(obj)) <= 80:
        return repr(obj)
    return _facts_line(obj)


def _stats(obj: object) -> str:
    """Диапазон значений числового массива или Series; для всего остального пустая строка."""
    values = obj.to_numpy() if isinstance(obj, pd.Series) else obj
    if not isinstance(values, np.ndarray) or values.size == 0:
        return ""
    if values.dtype.kind == "b":
        return f"true={int(values.sum())}"
    if values.dtype.kind not in "iuf":
        return ""
    finite = values[np.isfinite(values)]
    span = f"min={finite.min():.6g} max={finite.max():.6g}" if finite.size else "нет конечных значений"
    if values.dtype.kind != "f":
        return span
    return f"{span} nan={int(np.isnan(values).sum())} inf={int(np.isinf(values).sum())}"


def _preview(obj: object, limit: int) -> str:
    """Короткое текстовое превью: края массива, столбцы таблицы, элементы словаря или последовательности."""
    if isinstance(obj, pd.Series):
        obj = obj.to_numpy()
    if isinstance(obj, np.ndarray):
        return np.array2string(
            obj, threshold=2 * limit, edgeitems=max(limit // 2, 1), precision=6, max_line_width=100
        )
    if isinstance(obj, pd.DataFrame):
        return textwrap.fill("столбцы: " + ", ".join(map(str, obj.columns)), width=100)
    if isinstance(obj, dict):
        lines = [f"{key!r}: {_brief(value)}" for key, value in list(obj.items())[:limit]]
    elif isinstance(obj, (list, tuple)) and all(isinstance(item, _SCALARS) for item in obj):
        lines = [textwrap.fill(repr(list(obj[:limit])), width=100)]
    elif isinstance(obj, (list, tuple)):
        lines = [f"[{i}] {_brief(item)}" for i, item in enumerate(obj[:limit])]
    else:
        return ""
    if len(obj) > limit:
        lines.append(f"... ещё {len(obj) - limit}")
    return "\n".join(lines)


def peek(name: str, obj: object, limit: int = 6) -> None:
    """Сохранить ``obj`` в ``TRACE[name]`` и напечатать его тип, шейп, dtype, размер и превью.

    Parameters
    ----------
    name : str
        Ключ в ``TRACE``; по соглашению ``"<раздел>.<переменная>"``.
    obj : object
        Массив, таблица, контейнер, скаляр или любой другой промежуточный результат.
    limit : int, default 6
        Сколько элементов контейнера показывать; у массивов длиннее ``2 * limit`` показываются края.
    """
    TRACE[name] = obj
    if isinstance(obj, _SCALARS):
        print(f"[{name}] {type(obj).__name__} = {obj!r}")
        return
    stats = _stats(obj)
    print(f"[{name}] {_facts_line(obj)}" + (f" | {stats}" if stats else ""))
    for line in _preview(obj, limit).splitlines():
        print(f"    {line}")


def same(
    name: str, got: object, expected: object, rtol: float = 1e-10, atol: float = 1e-12
) -> None:
    """Проверить, что ``got`` совпадает с ``expected`` по шейпу и значениям, и напечатать наибольшую разницу.

    Parameters
    ----------
    name : str
        Название сверки.
    got, expected : array-like
        Сравниваемые значения; приводятся к массивам ``float``.
    rtol, atol : float
        Допуски ``numpy.testing.assert_allclose``.

    Raises
    ------
    AssertionError
        Если шейпы различаются или хотя бы один элемент выходит за допуск.
    """
    got = np.asarray(got, dtype=float)
    expected = np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        raise AssertionError(f"{name}: шейп {got.shape} != {expected.shape}")
    with np.errstate(invalid="ignore"):
        gap = np.abs(got - expected)
    finite = gap[np.isfinite(gap)]
    diff = float(finite.max()) if finite.size else 0.0
    try:
        np.testing.assert_allclose(got, expected, rtol=rtol, atol=atol)
    except AssertionError:
        print(f"[РАСХОЖДЕНИЕ] {name}: shape={got.shape}, max |разница| = {diff:.3e}")
        raise
    print(f"[ok] {name}: shape={got.shape}, max |разница| = {diff:.3e}")


def trace_index() -> pd.DataFrame:
    """Всё, что уже сохранено в ``TRACE``: имя, тип, шейп, dtype, число элементов, байты."""
    table = pd.DataFrame([{"name": name, **facts(obj)} for name, obj in TRACE.items()])
    return table.astype({"size": "Int64", "bytes": "Int64"})

### 1.3 Функции для просмотра графиков

График Plotly — контейнер массивов. `fig.data` — кортеж следов (traces), у каждого свои `x` и `y`; в `fig.layout.shapes` лежат линии разметки, в `fig.layout.annotations` — их подписи. Отладить график значит проверить, что эти массивы — те, которых вы ждёте.

- `figure_traces(fig)` возвращает строку на каждый след: имя, режим отрисовки, шейпы массивов и диапазоны значений.
- `figure_marks(fig)` возвращает линии разметки и подписи с их координатами. Система координат вида `y domain` означает, что значение — доля длины оси, а не значение данных.
- `trace_xy(fig, name)` возвращает нарисованные `x` и `y` одного следа в виде таблицы; безымянные следы выбираются по позиции.
- `describe_figure(name, fig)` сохраняет график в `TRACE` и печатает всё перечисленное.

In [3]:
def _span(values: np.ndarray) -> tuple[float, float]:
    """Наименьшее и наибольшее конечное значение; NaN, если конечных значений нет."""
    finite = values[np.isfinite(values)]
    return (float(finite.min()), float(finite.max())) if finite.size else (np.nan, np.nan)


def figure_traces(fig: go.Figure) -> pd.DataFrame:
    """Строка на каждый след ``fig``: имя, режим отрисовки, шейпы массивов и диапазоны значений."""
    rows = []
    for i, tr in enumerate(fig.data):
        x = np.asarray(tr.x, dtype=float)
        y = np.asarray(tr.y, dtype=float)
        (x_min, x_max), (y_min, y_max) = _span(x), _span(y)
        rows.append(
            {
                "след": i,
                "имя": tr.name,
                "тип": tr.type,
                "режим": tr.mode,
                "шейп x": x.shape,
                "шейп y": y.shape,
                "x min": x_min,
                "x max": x_max,
                "y min": y_min,
                "y max": y_max,
                "NaN в y": int(np.isnan(y).sum()),
                "цвет линии": tr.line.color,
                "толщина линии": tr.line.width,
                "заливка": tr.fill,
                "в легенде": tr.showlegend is not False,
            }
        )
    return pd.DataFrame(rows)


def figure_marks(fig: go.Figure) -> pd.DataFrame:
    """Линии разметки (``layout.shapes``) и подписи (``layout.annotations``) графика ``fig``."""
    shapes = [
        {
            "вид": f"shape/{s.type}",
            "xref": s.xref,
            "x0": s.x0,
            "x1": s.x1,
            "yref": s.yref,
            "y0": s.y0,
            "y1": s.y1,
            "цвет": s.line.color,
            "штрих": s.line.dash,
            "текст": None,
        }
        for s in fig.layout.shapes
    ]
    notes = [
        {
            "вид": "annotation",
            "xref": a.xref,
            "x0": a.x,
            "x1": a.x,
            "yref": a.yref,
            "y0": a.y,
            "y1": a.y,
            "цвет": None,
            "штрих": None,
            "текст": a.text,
        }
        for a in fig.layout.annotations
    ]
    return pd.DataFrame(shapes + notes)


def trace_xy(fig: go.Figure, name: str | int) -> pd.DataFrame:
    """``x`` и ``y`` одного следа; след выбирается по имени в легенде или по позиции в ``fig.data``.

    Raises
    ------
    RuntimeError
        Если ``name`` — строка и ей соответствует не ровно один след.
    """
    if isinstance(name, int):
        tr = fig.data[name]
    else:
        matches = [tr for tr in fig.data if tr.name == name]
        if len(matches) != 1:
            raise RuntimeError(f"{name}: следов {len(matches)}, ожидался один")
        tr = matches[0]
    return pd.DataFrame({"x": np.asarray(tr.x, dtype=float), "y": np.asarray(tr.y, dtype=float)})


def describe_figure(name: str, fig: go.Figure) -> None:
    """Сохранить ``fig`` в ``TRACE[name]`` и показать оформление, следы и линии разметки."""
    TRACE[name] = fig
    layout = fig.layout
    print(
        f"[{name}] Figure: следов {len(fig.data)}, shapes {len(layout.shapes)}, "
        f"annotations {len(layout.annotations)}"
    )
    print(f"    заголовок: {layout.title.text!r}")
    for axis_name, axis in (("x", layout.xaxis), ("y", layout.yaxis)):
        print(
            f"    ось {axis_name}: {axis.title.text!r}, range={axis.range}, "
            f"rangemode={axis.rangemode}"
        )
    print(f"    высота: {layout.height}")
    display(figure_traces(fig))
    marks = figure_marks(fig)
    if len(marks):
        display(marks)

### 1.4 Параметры

`DRONE_ID` выбирает строку, которая прослеживается. `PROBE_HOUR` — произвольный ранний момент; два других контрольных момента — текущий возраст дрона и этот возраст плюс `LOOKAHEAD_H`. Трёх точек хватает, чтобы напечатать каждый промежуточный результат целиком; разделы 12–15 переходят на сетки дашборда.

In [4]:
DRONE_ID = "DR-003057"  # любой drone_id из data/drone_survival.csv
PROBE_HOUR = 50.0  # ранний момент, в лётных часах
LOOKAHEAD_H = 100.0  # упреждение, которое прибавляется к возрасту дрона, в лётных часах

DATA_PATH = Path("data/drone_survival.csv")
ARTIFACTS_DIR = Path("artifacts")
MODELS_DIR = Path("models")

### 1.5 Метаданные прогона

`artifacts/meta.json` пишет последняя ячейка `drone_survival.ipynb`. В нём названы модель, выбранная на валидации (`best_name`), боевая модель, которая даёт прогнозы (`forecast_name`), и файлы моделей обоих этапов и лежат настройки, нужные дашборду: список признаков, словарь категорий, моменты времени, на которых считались метрики, горизонт и окно остаточного ресурса. В `artifacts/model_card.json` лежат результат проверки перед выпуском и происхождение прогона.

In [5]:
META = json.loads((ARTIFACTS_DIR / "meta.json").read_text())
peek("1.META", META, limit=len(META))
CARD = json.loads((ARTIFACTS_DIR / "model_card.json").read_text())
print(
    f"\nвыбрана на валидации: {META['best_name']}; прогнозы: боевая {META['forecast_name']}; "
    f"проверку перед выпуском прошли: {[kind for kind, result in CARD['gate']['kinds'].items() if result['passed']]}"
)
print(f"файлы оценочных моделей: {META['models']}; боевых: {META['production_models']}")

[1.META] dict size=21
    'format': 2
    'generated_at': '2026-11-19T11:44:43'
    'best_name': 'aft'
    'forecast_name': 'cox'
    'models': {'cox': 'best_cox.json', 'aft': 'best_aft.json'}
    'production_models': {'cox': 'production_cox.json', 'aft': 'production_aft.json'}
    'features': list size=16
    'cat_features': ['model', 'environment', 'is_refurbished', 'firmware_major']
    'categories': dict size=3
    'numeric_ranges': dict size=12
    'rul_horizon': 1111.0
    'rul_window': 100.0
    'split': dict size=5
    'n_trials': 60
    'n_boot': 1000
    'eval_times': dict size=2
    'calibration_horizons': [118.95, 219.12, 347.5]
    'landmark': 118.95
    'priority_thresholds': {'high': 0.5, 'medium': 0.2}
    'shap_base': {'cox': -0.044754818081855774, 'aft': -3.961398124694824}
    'shap_base_production': {'cox': -0.042453616857528687, 'aft': -3.7449607849121094}

выбрана на валидации: aft; прогнозы: боевая cox; проверку перед выпуском прошли: ['cox']
файлы оценочных моде

### 1.6 Объект модели

`Model.load` восстанавливает всё, что нужно для предсказания, а не только бустер:

| Атрибут | Содержимое | Где используется |
| --- | --- | --- |
| `kind` | `"cox"` или `"aft"` | каждая ветка ниже |
| `params` | подобранные параметры XGBoost; для AFT — распределение шума и его масштаб $\sigma$ | `_cum_hazard_at`, `hazard` |
| `features`, `cat_features` | столбцы таблицы признаков в порядке обучения | `_features_frame` |
| `categories` | словарь категорий, выученный на строках, на которых обучена модель (train + validation для боевой модели) | `encode_unknown_categories` |
| `booster` | деревья; `best_iteration` — последний используемый раунд: раунд ранней остановки у оценочной модели, последний сохранённый раунд у боевой | `_margin` |
| `_breslow` | только Cox: моменты отказов обучения и накопленная базовая опасность | `_cum_hazard_at`, `hazard` |
| `calibration` | параметры связи гамма-хрупкости или `None` | `_survival_at`, `hazard` |
| `history` | потери по раундам бустинга; у боевой модели только на обучении | только кривые обучения |

Для AFT margin — предсказанный $\log T$, а $\sigma$ — масштаб шума `aft_loss_distribution_scale`.

In [6]:
MODEL = Model.load(MODELS_DIR / META["production_models"]["aft"])
if MODEL.kind != "aft":
    raise RuntimeError(
        f"Основной проход написан для AFT, а production_models['aft'] в meta.json — {MODEL.kind}"
    )

dist = MODEL._aft_distribution
sigma = float(MODEL.params["aft_loss_distribution_scale"])
N_ROUNDS = MODEL.booster.num_boosted_rounds()
BEST_ITERATION = MODEL.booster.best_iteration
N_TREES_USED = BEST_ITERATION + 1
history_lengths = {
    f"{split}/{metric}": len(values)
    for split, metrics in MODEL.history.items()
    for metric, values in metrics.items()
}

peek("1.MODEL.params", MODEL.params, limit=len(MODEL.params))
peek("1.MODEL.features", MODEL.features, limit=len(MODEL.features))
peek("1.MODEL.cat_features", MODEL.cat_features)
peek("1.MODEL.categories", MODEL.categories)
peek("1.MODEL.calibration", MODEL.calibration)
peek("1.MODEL._breslow", MODEL._breslow)
peek("1.MODEL.history_lengths", history_lengths)

print(f"\nмодель: боевая {MODEL.kind} ({META['production_models']['aft']})")
print(f"распределение шума: {dist}, sigma={sigma:.6g}")
print(
    f"раундов бустинга сохранено: {N_ROUNDS}; best_iteration: {BEST_ITERATION}; "
    f"деревьев в предсказании: {N_TREES_USED}"
)
print(f"признаков у бустера: {MODEL.booster.num_features()}")

[1.MODEL.params] dict size=8
    'max_depth': 5
    'eta': 0.03159973167182181
    'min_child_weight': 14.606808553399388
    'subsample': 0.9793059603844647
    'colsample_bytree': 0.8022842436370485
    'reg_lambda': 1.618230350203502
    'aft_loss_distribution': 'logistic'
    'aft_loss_distribution_scale': 0.7381798504708477
[1.MODEL.features] list size=16
    ['model', 'environment', 'is_refurbished', 'initial_battery_cycles', 'firmware_major',
    'maintenance_interval_h', 'operator_experience_yrs', 'daily_utilization_h', 'avg_flight_min',
    'avg_payload_kg', 'payload_ratio', 'mean_temp_c', 'avg_humidity_pct', 'avg_wind_ms',
    'hard_landings_per_100h', 'mean_vibration_g']
[1.MODEL.cat_features] list size=4
    ['model', 'environment', 'is_refurbished', 'firmware_major']
[1.MODEL.categories] dict size=3
    'model': ['fixed_wing_vtol', 'hexa_heavy', 'quad_heavy', 'quad_light']
    'environment': ['coastal', 'desert', 'forest', 'urban']
    'firmware_major': [3, 4, 5]
[1.MODEL.

### 1.7 Строка дрона и контрольные моменты

Строка берётся из `data/drone_survival.csv` как есть: одна строка на дрон с его признаками, налётом на момент наблюдения (`duration_h`) и отметкой, закончилось ли наблюдение отказом (`event = 1`) или дрон ещё летал, когда его видели в последний раз (`event = 0`, цензурирование справа).

`AGE` — это `duration_h`. В `TIMES` три контрольных момента: `PROBE_HOUR`, `AGE` и `AGE + LOOKAHEAD_H`.

In [7]:
fleet = pd.read_csv(DATA_PATH)
drone = fleet.loc[fleet["drone_id"] == DRONE_ID].reset_index(drop=True)
if len(drone) != 1:
    raise ValueError(f"{DRONE_ID}: ожидалась одна строка, нашлось {len(drone)}")

AGE = float(drone["duration_h"].iloc[0])
EVENT = int(drone["event"].iloc[0])
TIMES = np.array([PROBE_HOUR, AGE, AGE + LOOKAHEAD_H])

peek("1.fleet", fleet)
peek("1.drone", drone)
peek("1.AGE", AGE)
peek("1.EVENT", EVENT)
peek("1.TIMES", TIMES)
status = "отказ наблюдался" if EVENT == 1 else "цензурирован, при последнем наблюдении ещё летал"
print(f"\n{DRONE_ID}: наблюдалось {AGE:g} ч, event={EVENT} ({status})")
drone.T

[1.fleet] DataFrame shape=(5000, 21) dtype=11 x float64, 5 x str, 4 x int64, 1 x bool size=105000 bytes=1021733
    столбцы: drone_id, model, environment, commission_date, is_refurbished, initial_battery_cycles,
    firmware_major, maintenance_interval_h, operator_experience_yrs, daily_utilization_h,
    avg_flight_min, avg_payload_kg, payload_ratio, mean_temp_c, avg_humidity_pct, avg_wind_ms,
    hard_landings_per_100h, mean_vibration_g, duration_h, event, failure_mode
[1.drone] DataFrame shape=(1, 21) dtype=11 x float64, 5 x str, 4 x int64, 1 x bool size=21 bytes=340
    столбцы: drone_id, model, environment, commission_date, is_refurbished, initial_battery_cycles,
    firmware_major, maintenance_interval_h, operator_experience_yrs, daily_utilization_h,
    avg_flight_min, avg_payload_kg, payload_ratio, mean_temp_c, avg_humidity_pct, avg_wind_ms,
    hard_landings_per_100h, mean_vibration_g, duration_h, event, failure_mode
[1.AGE] float = 193.5
[1.EVENT] int = 0
[1.TIMES] ndarray sha

,0
drone_id,DR-003057
model,quad_light
environment,urban
commission_date,2025-04-17
is_refurbished,False
initial_battery_cycles,24
firmware_major,3
maintenance_interval_h,200
operator_experience_yrs,1.44
daily_utilization_h,0.75


## 2. От сырой строки к margin

### 2.1 Таблица признаков

`Model._predict_margin` сначала вызывает `_features_frame`, которая делает три вещи:

1. оставляет только столбцы `features` в порядке обучения;
2. приводит категориальные столбцы к словарю обучения через `encode_unknown_categories`; уровень, которого не было в обучении, становится NaN (код категории −1), и в лог пишется предупреждение;
3. отвергает столбцы, dtype которых XGBoost принять не может.

`is_refurbished` числится в `cat_features`, но остаётся `bool`: `fit_categories` пропускает булевы столбцы, поэтому словаря у него нет. `duration_h`, `event` и идентификаторы отбрасываются здесь, поэтому предсказание от них зависеть не может.

Таблица показывает каждый признак до и после кодирования. `код категории` — позиция значения в словаре обучения.

In [8]:
raw_features = drone[MODEL.features]
vocabulary = {col: levels for col, levels in MODEL.categories.items() if col in MODEL.features}
X_hand = encode_unknown_categories(raw_features, vocabulary)
X = _features_frame(drone, MODEL.features, MODEL.categories)
pd.testing.assert_frame_equal(X, X_hand)
print("[ok] _features_frame = выбор столбцов + encode_unknown_categories")

dropped = sorted(set(drone.columns) - set(X.columns))
peek("2.raw_features", raw_features)
peek("2.vocabulary", vocabulary)
peek("2.X", X)
peek("2.dropped", dropped)

is_categorical = [isinstance(dtype, pd.CategoricalDtype) for dtype in X.dtypes]
feature_table = pd.DataFrame(
    {
        "исходный dtype": raw_features.dtypes.astype(str).to_numpy(),
        "dtype после кодирования": X.dtypes.astype(str).to_numpy(),
        "значение": [X[col].iloc[0] for col in X.columns],
        "код категории": pd.array(
            [
                int(X[col].cat.codes.iloc[0]) if categorical else None
                for col, categorical in zip(X.columns, is_categorical, strict=True)
            ],
            dtype="Int64",
        ),
        "словарь": [vocabulary.get(col) for col in X.columns],
    },
    index=X.columns,
)
feature_table

[ok] _features_frame = выбор столбцов + encode_unknown_categories
[2.raw_features] DataFrame shape=(1, 16) dtype=10 x float64, 3 x int64, 2 x str, 1 x bool size=16 bytes=270
    столбцы: model, environment, is_refurbished, initial_battery_cycles, firmware_major,
    maintenance_interval_h, operator_experience_yrs, daily_utilization_h, avg_flight_min,
    avg_payload_kg, payload_ratio, mean_temp_c, avg_humidity_pct, avg_wind_ms, hard_landings_per_100h,
    mean_vibration_g
[2.vocabulary] dict size=3
    'model': ['fixed_wing_vtol', 'hexa_heavy', 'quad_heavy', 'quad_light']
    'environment': ['coastal', 'desert', 'forest', 'urban']
    'firmware_major': [3, 4, 5]
[2.X] DataFrame shape=(1, 16) dtype=10 x float64, 3 x category, 2 x int64, 1 x bool size=16 bytes=389
    столбцы: model, environment, is_refurbished, initial_battery_cycles, firmware_major,
    maintenance_interval_h, operator_experience_yrs, daily_utilization_h, avg_flight_min,
    avg_payload_kg, payload_ratio, mean_temp_c, 

,исходный dtype,dtype после кодирования,значение,код категории,словарь
model,str,category,quad_light,3,"[fixed_wing_vtol, hexa_heavy, quad_heavy, quad..."
environment,str,category,urban,3,"[coastal, desert, forest, urban]"
is_refurbished,bool,bool,False,<NA>,None
initial_battery_cycles,int64,int64,24,<NA>,None
firmware_major,int64,category,3,0,"[3, 4, 5]"
maintenance_interval_h,int64,int64,200,<NA>,None
operator_experience_yrs,float64,float64,1.44,<NA>,None
daily_utilization_h,float64,float64,0.75,<NA>,None
avg_flight_min,float64,float64,23.5,<NA>,None
avg_payload_kg,float64,float64,0.722,<NA>,None


Неизвестный уровень предсказание не останавливает. Ячейка ниже копирует строку, подставляет в `model` уровень, которого нет в словаре, и кодирует его: предупреждение пишется в лог, значение становится NaN, а бустер всё равно возвращает margin. Если новый дрон получил неожиданное предсказание, это первое, что стоит проверить.

In [9]:
unseen = drone.copy()
unseen["model"] = "not_in_training"
X_unseen = _features_frame(unseen, MODEL.features, MODEL.categories)
margin_unseen = MODEL._margin(xgb.DMatrix(X_unseen, enable_categorical=True))
peek("2.X_unseen.model", X_unseen["model"])
peek("2.margin_unseen", margin_unseen)
print(f"код категории у неизвестного уровня: {int(X_unseen['model'].cat.codes.iloc[0])}")

model: 1 rows with unseen categories ['not_in_training'] set to NaN


[2.X_unseen.model] Series shape=(1,) dtype=category size=1 bytes=210
    [nan]
[2.margin_unseen] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=4.62486 max=4.62486 nan=0 inf=0
    [4.624861]
код категории у неизвестного уровня: -1


### 2.2 DMatrix

Закодированная таблица оборачивается в `xgb.DMatrix` с `enable_categorical=True` и **без меток**: `duration_h` и `event` нужны только при обучении. Имена признаков матрицы должны совпадать с теми, на которых обучен бустер, а таблица позволяет сравнить типы.

Коды типов XGBoost: `c` — категориальный, `i` — индикатор (булев), `int` — целый, `float` — вещественный.

In [10]:
dm = xgb.DMatrix(X, enable_categorical=True)
dm_labels = dm.get_label()
peek("2.dm", dm)
peek("2.dm_labels", dm_labels)
if dm_labels.size:
    raise RuntimeError("прогнозный DMatrix не должен содержать меток")
if dm.feature_names != MODEL.booster.feature_names:
    raise RuntimeError("имена признаков в DMatrix и в бустере не совпадают")
print(f"DMatrix: строк {dm.num_row()}, столбцов {dm.num_col()}, меток {dm_labels.size}")
print("столбцы строки, которые не вошли в матрицу:", dropped)

pd.DataFrame(
    {
        "dtype в pandas": X.dtypes.astype(str).to_numpy(),
        "тип в DMatrix": dm.feature_types,
        "тип в бустере": MODEL.booster.feature_types,
    },
    index=dm.feature_names,
)

[2.dm] DMatrix shape=(1, 16) size=16
[2.dm_labels] ndarray shape=(0,) dtype=float32 size=0 bytes=0
    []
DMatrix: строк 1, столбцов 16, меток 0
столбцы строки, которые не вошли в матрицу: ['commission_date', 'drone_id', 'duration_h', 'event', 'failure_mode']


,dtype в pandas,тип в DMatrix,тип в бустере
model,category,c,c
environment,category,c,c
is_refurbished,bool,i,i
initial_battery_cycles,int64,int,int
firmware_major,category,c,c
maintenance_interval_h,int64,int,int
operator_experience_yrs,float64,float,float
daily_utilization_h,float64,float,float
avg_flight_min,float64,float,float
avg_payload_kg,float64,float,float


### 2.3 Margin

`Model._margin` — единственное место, где XGBoost делает предсказание:

$$\hat f(x) = \texttt{booster.predict}(\texttt{output\_margin=True},\ \texttt{iteration\_range}=(0,\ \texttt{best\_iteration}+1)).$$

При отладке важны три детали:

- `output_margin=True` возвращает сырую сумму деревьев. Без него XGBoost для обеих целевых функций выживаемости применяет $\exp$; ячейка проверяет, что результат равен $\exp(\hat f)$.
- `iteration_range` ограничивает сумму `best_iteration + 1` деревьями. В оценочной модели (`best_*.json`) лежат и раунды, обученные после раунда ранней остановки, поэтому для неё вызов `booster.predict(dm, output_margin=True)` без `iteration_range` лучшей итерацией не ограничен. Боевая модель обучена ровно на это число раундов, поэтому здесь три результата совпадают; ячейка печатает результат вызова без `iteration_range` рядом с margin по всем сохранённым раундам и с margin, который использует `Model`.
- Результат имеет тип `float32` и шейп `(n,)`: один margin на строку.

Для AFT $\hat f(x)$ — предсказанный $\log T$. $\exp(\hat f)$ — момент, в котором стандартизованный логарифм времени $z$ из раздела 4 равен нулю; раздел 5 считает $S$ в этой точке.

In [11]:
iteration_range = (0, MODEL.booster.best_iteration + 1)
direct = MODEL.booster.predict(dm, output_margin=True, iteration_range=iteration_range)
via_margin = MODEL._margin(dm)
margin = MODEL._predict_margin(drone)
same("booster.predict = _margin", direct, via_margin)
same("_margin = _predict_margin", via_margin, margin)

transformed = MODEL.booster.predict(dm, iteration_range=iteration_range)
same("predict без output_margin = exp(margin)", transformed, np.exp(margin), rtol=1e-6)

margin_all_rounds = MODEL.booster.predict(
    dm, output_margin=True, iteration_range=(0, N_ROUNDS)
)
margin_bare = MODEL.booster.predict(dm, output_margin=True)

peek("2.margin", margin)
peek("2.transformed", transformed)
peek("2.margin_all_rounds", margin_all_rounds)
peek("2.margin_bare", margin_bare)
print(f"\nmargin = предсказанный log T = {margin[0]:.6f}")
print(f"exp(margin) = {np.exp(margin[0]):.2f} ч")
print(
    f"margin по всем {N_ROUNDS} раундам = {margin_all_rounds[0]:.6f}; "
    f"booster.predict без iteration_range = {margin_bare[0]:.6f}; "
    f"этот вызов минус margin, который использует Model = {margin_bare[0] - margin[0]:+.6f}"
)

[ok] booster.predict = _margin: shape=(1,), max |разница| = 0.000e+00
[ok] _margin = _predict_margin: shape=(1,), max |разница| = 0.000e+00
[ok] predict без output_margin = exp(margin): shape=(1,), max |разница| = 0.000e+00
[2.margin] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=4.57009 max=4.57009 nan=0 inf=0
    [4.570089]
[2.transformed] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=96.5527 max=96.5527 nan=0 inf=0
    [96.55269]
[2.margin_all_rounds] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=4.57009 max=4.57009 nan=0 inf=0
    [4.570089]
[2.margin_bare] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=4.57009 max=4.57009 nan=0 inf=0
    [4.570089]

margin = предсказанный log T = 4.570089
exp(margin) = 96.55 ч
margin по всем 346 раундам = 4.570089; booster.predict без iteration_range = 4.570089; этот вызов минус margin, который использует Model = +0.000000


### 2.4 Margin по деревьям

Margin — это свободный член плюс одно значение листа на дерево:

$$\hat f(x) = b + \sum_{j=0}^{T-1} w_j(x), \qquad T = \texttt{best\_iteration} + 1.$$

- $b$ получается из `base_score` бустера, который читается из `booster.save_config()`. Здесь он входит в margin как $b = \log(\texttt{base\_score})$; первая сверка ячейки подтверждает это для загруженной модели.
- $w_j(x)$ — значение листа, в который строка $x$ попадает в дереве $j$. Предсказание с `iteration_range=(j, j + 1)` возвращает $b + w_j(x)$.
- `pred_leaf=True` возвращает индекс этого листа в каждом дереве, шейп `(n, T)`.
- Поэтапный margin после $j + 1$ деревьев, `iteration_range=(0, j + 1)`, показывает, как предсказание сходится по раундам бустинга.

XGBoost считает в `float32`, поэтому суммы, собранные здесь заново, совпадают с ним примерно до $10^{-6}$, а не до машинной точности.

In [12]:
booster_config = json.loads(MODEL.booster.save_config())
base_score_raw = booster_config["learner"]["learner_model_param"]["base_score"]
base_score = float(np.atleast_1d(json.loads(base_score_raw))[0])
intercept = float(np.log(base_score))

tree_index = np.arange(N_TREES_USED)
staged_margin = np.array(
    [
        MODEL.booster.predict(dm, output_margin=True, iteration_range=(0, j + 1))[0]
        for j in tree_index
    ],
    dtype=float,
)
single_tree = np.array(
    [
        MODEL.booster.predict(dm, output_margin=True, iteration_range=(j, j + 1))[0]
        for j in tree_index
    ],
    dtype=float,
)
tree_values = single_tree - intercept
leaf_index = MODEL.booster.predict(dm, pred_leaf=True, iteration_range=iteration_range)

peek("2.base_score", base_score)
peek("2.intercept", intercept)
peek("2.staged_margin", staged_margin)
peek("2.single_tree", single_tree)
peek("2.tree_values", tree_values)
peek("2.leaf_index", leaf_index)

same(
    "свободный член + накопленная сумма значений листьев = поэтапный margin",
    intercept + np.cumsum(tree_values),
    staged_margin,
    rtol=0,
    atol=1e-4,
)
same("поэтапный margin после последнего дерева = margin", staged_margin[-1:], margin)

tree_table = pd.DataFrame(
    {
        "дерево": tree_index,
        "индекс листа": leaf_index[0].astype(int),
        "значение листа": tree_values,
        "поэтапный margin": staged_margin,
    }
)
by_size = tree_table["значение листа"].abs().sort_values(ascending=False).index
print("\nпервые и последние деревья:")
display(pd.concat([tree_table.head(5), tree_table.tail(5)]))
print("деревья, которые сильнее всего сдвигают margin:")
display(tree_table.loc[by_size].head(5))

[2.base_score] float = 0.5
[2.intercept] float = -0.6931471805599453
[2.staged_margin] ndarray shape=(346,) dtype=float64 size=346 bytes=2768 | min=4.56539 max=6.20783 nan=0 inf=0
    [6.207828 6.193474 6.156084 ... 4.58793  4.565393 4.570089]
[2.single_tree] ndarray shape=(346,) dtype=float64 size=346 bytes=2768 | min=-0.74738 max=6.20783 nan=0 inf=0
    [ 6.207828 -0.7075   -0.730538 ... -0.693142 -0.715684 -0.688451]
[2.tree_values] ndarray shape=(346,) dtype=float64 size=346 bytes=2768 | min=-0.054233 max=6.90097 nan=0 inf=0
    [ 6.900975e+00 -1.435322e-02 -3.739041e-02 ...  5.660537e-06 -2.253658e-02  4.696010e-03]
[2.leaf_index] ndarray shape=(1, 346) dtype=float32 size=346 bytes=1384 | min=1 max=49 nan=0 inf=0
    [[ 1. 22.  8. ... 22. 35.  9.]]
[ok] свободный член + накопленная сумма значений листьев = поэтапный margin: shape=(346,), max |разница| = 2.281e-06
[ok] поэтапный margin после последнего дерева = margin: shape=(1,), max |разница| = 0.000e+00

первые и последние дерев

,дерево,индекс листа,значение листа,поэтапный margin
0,0,1,6.90097,6.20783
1,1,22,-0.0143532,6.19347
2,2,8,-0.0373904,6.15608
3,3,19,-0.0389404,6.11714
4,4,27,-0.00469047,6.11245
341,341,24,-0.000157358,4.58893
342,342,24,-0.00100857,4.58792
343,343,22,5.66054e-06,4.58793
344,344,35,-0.0225366,4.56539
345,345,9,0.00469601,4.57009


деревья, которые сильнее всего сдвигают margin:


,дерево,индекс листа,значение листа,поэтапный margin
0,0,1,6.90097,6.20783
6,6,39,-0.054233,6.0228
9,9,13,-0.0474762,5.95766
3,3,19,-0.0389404,6.11714
11,11,15,-0.0376016,5.89416


### 2.5 Margin по признакам

`pred_contribs=True` возвращает вклады TreeSHAP с шейпом `(n, p + 1)`: столбец на признак плюс последний столбец смещения. Каждая строка в сумме даёт margin:

$$\hat f(x) = \phi_0 + \sum_{i=1}^{p} \phi_i(x).$$

Смещение $\phi_0$ — базовое значение SHAP, ожидаемый margin до того, как известен хоть один признак. Это **не** свободный член $b$ из раздела 2.4; ячейка печатает оба числа.

`Model.explain` возвращает те же числа для скора `risk`, поэтому для AFT знаки перевёрнуты (раздел 3): положительное значение сдвигает к более раннему отказу. Его базовое значение — то, что в `meta.json` хранится как `shap_base`, начальная точка водопадной диаграммы дашборда.

In [13]:
contribs = MODEL.booster.predict(dm, pred_contribs=True, iteration_range=iteration_range)
feature_contribs = contribs[:, :-1]
bias = contribs[:, -1]
peek("2.contribs", contribs)
peek("2.feature_contribs", feature_contribs)
peek("2.bias", bias)
same("смещение + сумма вкладов = margin", contribs.sum(axis=1), margin, rtol=0, atol=1e-4)

explanation = MODEL.explain(drone)
peek("2.explanation.values", explanation.values)
peek("2.explanation.base_values", explanation.base_values)
same("значения explain = -вклады", explanation.values, -feature_contribs, rtol=0, atol=1e-5)
same("базовое значение explain = -смещение", explanation.base_values, -bias, rtol=0, atol=1e-5)
same(
    "базовое значение explain = shap_base в meta.json",
    explanation.base_values,
    np.array([META["shap_base_production"][MODEL.kind]]),
    rtol=0,
    atol=1e-5,
)
print(f"\nсвободный член b = {intercept:.6f}; смещение SHAP = {bias[0]:.6f}")

contrib_table = pd.DataFrame(
    {
        "значение": [X[col].iloc[0] for col in X.columns],
        "вклад в margin": feature_contribs[0],
        "вклад в risk": explanation.values[0],
    },
    index=X.columns,
)
by_size = contrib_table["вклад в margin"].abs().sort_values(ascending=False).index
contrib_table.loc[by_size]

[2.contribs] ndarray shape=(1, 17) dtype=float32 size=17 bytes=68 | min=-1.11215 max=3.74496 nan=0 inf=0
    [[-0.011641 -0.009544  0.030563 ... -0.179266 -0.03675   3.744961]]
[2.feature_contribs] ndarray shape=(1, 16) dtype=float32 size=16 bytes=64 | min=-1.11215 max=2.31204 nan=0 inf=0
    [[-0.011641 -0.009544  0.030563 ...  2.312041 -0.179266 -0.03675 ]]
[2.bias] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=3.74496 max=3.74496 nan=0 inf=0
    [3.744961]
[ok] смещение + сумма вкладов = margin: shape=(1,), max |разница| = 1.431e-06
[2.explanation.values] ndarray shape=(1, 16) dtype=float32 size=16 bytes=64 | min=-2.31204 max=1.11215 nan=0 inf=0
    [[ 0.011641  0.009544 -0.030563 ... -2.312041  0.179266  0.03675 ]]
[2.explanation.base_values] ndarray shape=(1,) dtype=float64 size=1 bytes=8 | min=-3.74496 max=-3.74496 nan=0 inf=0
    [-3.744961]
[ok] значения explain = -вклады: shape=(1, 16), max |разница| = 0.000e+00
[ok] базовое значение explain = -смещение: shape=(1,), ma

,значение,вклад в margin,вклад в risk
avg_wind_ms,6.28,2.31204,-2.31204
mean_temp_c,0.3,-1.11215,1.11215
hard_landings_per_100h,5.02,-0.179266,0.179266
firmware_major,3,-0.152244,0.152244
initial_battery_cycles,24,0.0955785,-0.0955785
avg_humidity_pct,48.7,0.0869008,-0.0869008
avg_flight_min,23.5,0.0740775,-0.0740775
operator_experience_yrs,1.44,-0.0716829,0.0716829
maintenance_interval_h,200,-0.069208,0.069208
daily_utilization_h,0.75,-0.0678706,0.0678706


### 2.6 Предсказание не зависит от `duration_h` и `event`

Ни одного из этих столбцов в таблице признаков нет, поэтому их изменение не должно трогать ни margin, ни кривую выживаемости. Именно это позволяет запрашивать $S(t)$ в любой момент $t$, в том числе за пределами текущего возраста дрона.

In [14]:
shifted = drone.copy()
shifted["duration_h"] = 1.0
shifted["event"] = 1
same("margin не зависит от duration_h и event", MODEL._predict_margin(shifted), margin)
same(
    "survival не зависит от duration_h и event",
    MODEL.survival(shifted, TIMES),
    MODEL.survival(drone, TIMES),
)

[ok] margin не зависит от duration_h и event: shape=(1,), max |разница| = 0.000e+00
[ok] survival не зависит от duration_h и event: shape=(1, 3), max |разница| = 0.000e+00


## 3. Risk

`risk` — ранжирующий скор: большее значение означает более ранний ожидаемый отказ. У Cox это сам margin, логарифм относительного риска. У AFT знак меняется, потому что большой $\log T$ — это поздний отказ:

$$\mathrm{risk}(x) = -\hat f(x).$$

У двух моделей разные шкалы (логарифм относительного риска против логарифма часов), поэтому значения `risk` сравнимы только внутри одной модели. Это скор, который ранжирует C-index (раздел 10) и по которому нарезается `risk_group` в таблице предсказаний.

In [15]:
risk = MODEL.risk(drone)
same("risk = -margin", risk, -margin)
peek("3.risk", risk)
print(f"risk = -margin = {risk[0]:.6f}")

[ok] risk = -margin: shape=(1,), max |разница| = 0.000e+00
[3.risk] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=-4.57009 max=-4.57009 nan=0 inf=0
    [-4.570089]
risk = -margin = -4.570089


## 4. Накопленная опасность

`_cum_hazard_at` возвращает $H(t \mid x) = -\log S(t \mid x)$ до калибровки.

Для AFT время входит через стандартизованный логарифм времени

$$z = (\log t - \hat f(x)) / \sigma,$$

а `_aft_noise` возвращает $H(z)$ и её производную $H'(z)$ для распределения шума:

| шум | время до отказа | $H(z)$ | $H'(z)$ |
| --- | --- | --- | --- |
| normal | логнормальное | $-\log(1-\Phi(z))$ | $\varphi(z)/(1-\Phi(z))$ |
| logistic | лог-логистическое | $\log(1+e^{z})$ | $\mathrm{expit}(z)$ |
| extreme | Вейбулл | $e^{z}$ | $e^{z}$ |

Шейпы: `np.log(TIMES)` имеет шейп `(m,)`, а `margin[:, None]` — `(n, 1)`, поэтому $z$ после broadcasting имеет шейп `(n, m)`. `_cum_hazard_at` принимает моменты времени как `(m,)`, общие для всех строк, или как `(n, m)`, свой набор на строку; возвращает она всегда `(n, m)`.

При $t = 0$ имеем $\log t = -\infty$, поэтому $z = -\infty$, $H = 0$ и $S = 1$: каждая кривая начинается с 1.

In [16]:
log_t = np.log(TIMES)
margin_col = margin[:, None]
z = (log_t - margin_col) / sigma
H_noise, dH_dz = _aft_noise(dist, z)
peek("4.log_t", log_t)
peek("4.margin_col", margin_col)
peek("4.z", z)
peek("4.H_noise", H_noise)
peek("4.dH_dz", dH_dz)

if dist == "logistic":
    H_closed, dH_closed = np.logaddexp(0.0, z), expit(z)
elif dist == "normal":
    log_sf = norm.logsf(z)
    H_closed = -log_sf
    dH_closed = np.exp(norm.logpdf(z) - log_sf)
else:
    H_closed = np.exp(z)
    dH_closed = H_closed
same("H(z) = замкнутая форма", H_noise, H_closed)
same("H'(z) = замкнутая форма", dH_dz, dH_closed)

H = MODEL._cum_hazard_at(margin, TIMES)
peek("4.H", H)
same("_cum_hazard_at", H, H_noise)

H_at_zero = MODEL._cum_hazard_at(margin, np.array([0.0]))
peek("4.H_at_zero", H_at_zero)
same("H(0) = 0", H_at_zero, np.zeros((len(drone), 1)))

pd.DataFrame({"t": TIMES, "log t": log_t, "z": z[0], "H(z)": H[0], "H'(z)": dH_dz[0]})

[4.log_t] ndarray shape=(3,) dtype=float64 size=3 bytes=24 | min=3.91202 max=5.68188 nan=0 inf=0
    [3.912023 5.265278 5.681878]
[4.margin_col] ndarray shape=(1, 1) dtype=float32 size=1 bytes=4 | min=4.57009 max=4.57009 nan=0 inf=0
    [[4.570089]]
[4.z] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=-0.891471 max=1.50612 nan=0 inf=0
    [[-0.891471  0.941761  1.506122]]
[4.H_noise] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.343627 max=1.70642 nan=0 inf=0
    [[0.343627 1.271022 1.706421]]
[4.dH_dz] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.290806 max=0.818486 nan=0 inf=0
    [[0.290806 0.719455 0.818486]]
[ok] H(z) = замкнутая форма: shape=(1, 3), max |разница| = 0.000e+00
[ok] H'(z) = замкнутая форма: shape=(1, 3), max |разница| = 0.000e+00
[4.H] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.343627 max=1.70642 nan=0 inf=0
    [[0.343627 1.271022 1.706421]]
[ok] _cum_hazard_at: shape=(1, 3), max |разница| = 0.000e+00
[4.H_at_zer

,t,log t,z,H(z),H'(z)
0,50,3.91202,-0.891471,0.343627,0.290806
1,193.5,5.26528,0.941761,1.27102,0.719455
2,293.5,5.68188,1.50612,1.70642,0.818486


## 5. Выживаемость

`_survival_at` превращает накопленную опасность в вероятность того, что в момент $t$ дрон ещё работает:

$$S(t \mid x) = \exp(-H).$$

Если `calibration` задана, $H$ сначала проходит через связь гамма-хрупкости. При дисперсии $v \ge 10^{-8}$

$$\Lambda = \log(1 + v \, k \, H) / v,$$

а в пределе $v \to 0$ получается $\Lambda = kH$. Связь поправляет кривые на различия между дронами, которые признаки не улавливают. `drone_survival.ipynb` сбрасывает `calibration` в `None`, когда подобранная дисперсия равна нулю, и модель остаётся такой, какой обучена. `survival` делает то же, что `_survival_at`, но сама считает margin.

Ячейка печатает, по какой ветке идёт загруженная модель. Для самопроверки она ещё считает $S$ в точке $\exp(\hat f)$, где $z = 0$: для нормального или логистического шума без калибровки это 0.5, то есть $\exp(\hat f)$ — медианное время до отказа.

In [17]:
if MODEL.calibration is None:
    H_used = H
    print("calibration is None: накопленная опасность входит в exp(-H) без изменений")
else:
    v = MODEL.calibration["frailty_variance"]
    k = MODEL.calibration["scale"]
    H_used = k * H if v < 1e-8 else np.log1p(v * k * H) / v
    H_linked, _ = _frailty_link(H, **MODEL.calibration)
    same("связь хрупкости", H_used, H_linked)
    print(f"H заменено на log(1 + v*k*H) / v при v={v:.4g}, k={k:.4g}")

S_hand = np.exp(-H_used)
S_at = MODEL._survival_at(margin, TIMES)
S = MODEL.survival(drone, TIMES)
peek("5.H_used", H_used)
peek("5.S", S)
same("_survival_at", S_at, S_hand)
same("survival", S, S_hand)

t_at_z0 = np.exp(margin).astype(float)
S_at_z0 = MODEL.survival(drone, t_at_z0)
peek("5.S_at_z0", S_at_z0)
print(f"\nS в точке exp(margin) = {t_at_z0[0]:.2f} ч: {S_at_z0[0, 0]:.6f}")
pd.DataFrame({"t": TIMES, "H": H_used[0], "S": S[0]})

calibration is None: накопленная опасность входит в exp(-H) без изменений
[5.H_used] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.343627 max=1.70642 nan=0 inf=0
    [[0.343627 1.271022 1.706421]]
[5.S] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.181514 max=0.709194 nan=0 inf=0
    [[0.709194 0.280545 0.181514]]
[ok] _survival_at: shape=(1, 3), max |разница| = 0.000e+00
[ok] survival: shape=(1, 3), max |разница| = 0.000e+00
[5.S_at_z0] ndarray shape=(1, 1) dtype=float64 size=1 bytes=8 | min=0.5 max=0.5 nan=0 inf=0
    [[0.5]]

S в точке exp(margin) = 96.55 ч: 0.500000


,t,H,S
0,50,0.343627,0.709194
1,193.5,1.27102,0.280545
2,293.5,1.70642,0.181514


## 6. Условная выживаемость

Возраст $a$ — уже налётанные часы. Вероятность пролететь ещё $u$ часов при неизменных признаках:

$$S(a+u \mid x) \,/\, S(a \mid x).$$

При $u = 0$ отношение равно 1. При $u = 100$ это вероятность прожить следующие 100 часов. Там, где $S(a) = 0$, функция возвращает NaN; на конечном возрасте модель AFT такого не даёт.

Шейпы: в `ages` один возраст на строку, и он становится столбцом `(n, 1)`; упреждения `u` общие для всех строк и становятся строкой `(1, m)`; их сумма — матрица `(n, m)` абсолютных моментов времени, в которых считается $S$.

In [18]:
U = np.array([0.0, LOOKAHEAD_H])
ages = np.array([AGE])
ages_col = ages[:, None]
u_row = U[None, :]
future = ages_col + u_row
s_age = MODEL._survival_at(margin, ages_col)
s_ahead = MODEL._survival_at(margin, future)
ratio = s_ahead / s_age
cond = MODEL.conditional_survival(drone, ages, U)
peek("6.ages_col", ages_col)
peek("6.u_row", u_row)
peek("6.future", future)
peek("6.s_age", s_age)
peek("6.s_ahead", s_ahead)
peek("6.cond", cond)
same("conditional_survival", cond, ratio)

pd.DataFrame(
    {
        "u": U,
        "a + u": future[0],
        "S(a)": np.full(len(U), s_age[0, 0]),
        "S(a+u)": s_ahead[0],
        "S(a+u)/S(a)": cond[0],
    }
)

[6.ages_col] ndarray shape=(1, 1) dtype=float64 size=1 bytes=8 | min=193.5 max=193.5 nan=0 inf=0
    [[193.5]]
[6.u_row] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=0 max=100 nan=0 inf=0
    [[  0. 100.]]
[6.future] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=193.5 max=293.5 nan=0 inf=0
    [[193.5 293.5]]
[6.s_age] ndarray shape=(1, 1) dtype=float64 size=1 bytes=8 | min=0.280545 max=0.280545 nan=0 inf=0
    [[0.280545]]
[6.s_ahead] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=0.181514 max=0.280545 nan=0 inf=0
    [[0.280545 0.181514]]
[6.cond] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=0.647006 max=1 nan=0 inf=0
    [[1.       0.647006]]
[ok] conditional_survival: shape=(1, 2), max |разница| = 0.000e+00


,u,a + u,S(a),S(a+u),S(a+u)/S(a)
0,0,193.5,0.280545,0.280545,1
1,100,293.5,0.280545,0.181514,0.647006


## 7. Интенсивность

Интенсивность — скорость отказов на один лётный час среди дронов, которые в момент $t$ ещё работают. Для AFT это производная опасности шума, переведённая из шкалы $z$ в часы:

$$h(t \mid x) = H'(z) \,/\, (\sigma \, t).$$

Если `calibration` задана, результат умножается на наклон связи $k / (1 + v k H)$. Ячейка печатает, какой из двух случаев имеет место.

Независимая проверка следует из определения: интенсивность — производная накопленной опасности, $h(t) = \mathrm{d}[-\log S(t)]/\mathrm{d}t$. Ячейка численно дифференцирует `survival` центральной разностью и сравнивает. Эта проверка покрывает и связь, потому что `survival` её уже включает.

In [19]:
h_noise = dH_dz / (sigma * TIMES)
if MODEL.calibration is None:
    link_slope = np.ones_like(H)
    print("калибровки нет: множитель наклона равен 1")
else:
    _, link_slope = _frailty_link(H, **MODEL.calibration)
    print("интенсивность умножена на наклон связи")
h_hand = link_slope * h_noise
h = MODEL.hazard(drone, TIMES)
peek("7.h_noise", h_noise)
peek("7.link_slope", link_slope)
peek("7.h", h)
same("hazard", h, h_hand)

step_h = 1e-3
H_after = -np.log(MODEL.survival(drone, TIMES + step_h))
H_before = -np.log(MODEL.survival(drone, TIMES - step_h))
h_numeric = (H_after - H_before) / (2 * step_h)
peek("7.h_numeric", h_numeric)
same("hazard = d[-log S]/dt, центральная разность", h, h_numeric, rtol=1e-6, atol=0)

pd.DataFrame(
    {
        "t": TIMES,
        "H'(z)": dH_dz[0],
        "h, на час": h[0],
        f"h, на {HAZARD_SCALE:g} ч": HAZARD_SCALE * h[0],
    }
)

калибровки нет: множитель наклона равен 1
[7.h_noise] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.00377782 max=0.00787901 nan=0 inf=0
    [[0.007879 0.005037 0.003778]]
[7.link_slope] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=1 max=1 nan=0 inf=0
    [[1. 1. 1.]]
[7.h] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.00377782 max=0.00787901 nan=0 inf=0
    [[0.007879 0.005037 0.003778]]
[ok] hazard: shape=(1, 3), max |разница| = 0.000e+00
[7.h_numeric] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.00377782 max=0.00787901 nan=0 inf=0
    [[0.007879 0.005037 0.003778]]
[ok] hazard = d[-log S]/dt, центральная разность: shape=(1, 3), max |разница| = 3.984e-13


,t,H'(z),"h, на час","h, на 100 ч"
0,50,0.290806,0.00787901,0.787901
1,193.5,0.719455,0.00503687,0.503687
2,293.5,0.818486,0.00377782,0.377782


## 8. Запись наблюдения `_surv`

`_surv` упаковывает `event` и `duration_h` в структурный массив записей `(event, time)` — формат, которого ждут метрики `sksurv`. Это **наблюдение**, а не предсказание.

На выборке метрики сравнивают предсказания с такими записями. Brier score и AUC сравнивают `survival` на горизонте с тем, отказал ли дрон к этому моменту. C-index проверяет, что в паре дронов тот, что отказал раньше, получил больший `risk`. Одна строка сама по себе ни из чего не вычитается; раздел 10 выполняет сравнение на всей тестовой таблице.

`event = False` означает, что запись цензурирована: в момент `time` дрон ещё летал.

In [20]:
label = _surv(drone)
peek("8.label", label)
print(f"поля: {label.dtype.names}; {label.dtype.itemsize} байт на запись")
print(f"запись: event={bool(label['event'][0])}, time={float(label['time'][0]):g} ч")

s_now = float(MODEL.survival(drone, ages)[0, 0])
peek("8.s_now", s_now)
print(f"S({AGE:g}) = {s_now:.6f}: значение кривой в момент этой записи")
print(f"risk = {risk[0]:.6f}: скор, который C-index сравнивает между записями")

[8.label] ndarray shape=(1,) dtype=[('event', '?'), ('time', '<f8')] size=1 bytes=9
    [(False, 193.5)]
поля: ('event', 'time'); 9 байт на запись
запись: event=False, time=193.5 ч
[8.s_now] float = 0.2805448604250952
S(193.5) = 0.280545: значение кривой в момент этой записи
risk = -4.570089: скор, который C-index сравнивает между записями


## 9. Та же строка на модели Cox

### 9.1 Модель, margin и risk

Модель Cox читается из файла, названного в `meta.json`. Её margin — логарифм относительного риска, и `risk` равен ему без смены знака. $\exp(\hat f)$ — относительный риск этого дрона по сравнению с базовой линией.

In [21]:
COX = Model.load(MODELS_DIR / META["production_models"]["cox"])
peek("9.COX.params", COX.params, limit=len(COX.params))
peek("9.COX.calibration", COX.calibration)
peek("9.COX._breslow", COX._breslow)
print(
    f"\nkind={COX.kind}; раундов бустинга сохранено: {COX.booster.num_boosted_rounds()}; "
    f"best_iteration: {COX.booster.best_iteration}"
)

cox_margin = COX._predict_margin(drone)
cox_risk = COX.risk(drone)
hazard_ratio = np.exp(cox_margin)
peek("9.cox_margin", cox_margin)
peek("9.hazard_ratio", hazard_ratio)
same("Cox risk = margin", cox_risk, cox_margin)
print(f"margin = логарифм относительного риска = {cox_margin[0]:.6f}")
print(f"относительный риск exp(margin) = {hazard_ratio[0]:.4f}")

[9.COX.params] dict size=6
    'max_depth': 3
    'eta': 0.10161123659069494
    'min_child_weight': 2.31735485138156
    'subsample': 0.7835346123945779
    'colsample_bytree': 0.9582043558653918
    'reg_lambda': 1.8777925383998129
[9.COX.calibration] dict size=2
    'frailty_variance': 0.2678234346812382
    'scale': 1.0453121212260807
[9.COX._breslow] tuple size=2
    [0] ndarray shape=(2543,) dtype=float64 size=2543 bytes=20344
    [1] ndarray shape=(2543,) dtype=float64 size=2543 bytes=20344

kind=cox; раундов бустинга сохранено: 118; best_iteration: 117
[9.cox_margin] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=1.67805 max=1.67805 nan=0 inf=0
    [1.678049]
[9.hazard_ratio] ndarray shape=(1,) dtype=float32 size=1 bytes=4 | min=5.3551 max=5.3551 nan=0 inf=0
    [5.355099]
[ok] Cox risk = margin: shape=(1,), max |разница| = 0.000e+00
margin = логарифм относительного риска = 1.678049
относительный риск exp(margin) = 5.3551


### 9.2 Базовая линия Бреслоу и накопленная опасность

`COX._breslow` — пара массивов длины `k`: различные моменты отказов строк, на которых обучена модель (здесь train + validation), и накопленная базовая опасность $H_0$ в каждом из них. $H_0$ — ступенчатая функция: в момент $t$ она принимает значение в последнем моменте отказа обучения не позже $t$, а до первого такого момента равна 0. Тогда

$$H(t \mid x) = \exp(\hat f(x)) \, H_0(t).$$

`np.searchsorted(ev_times, t, side="right")` считает моменты отказов $\le t$; вычитание 1 даёт индекс действующей ступеньки.

In [22]:
ev_times, h0 = COX._breslow
peek("9.ev_times", ev_times)
peek("9.h0", h0)
print(
    f"моменты отказов строго возрастают: {bool(np.all(np.diff(ev_times) > 0))}; "
    f"H0 не убывает: {bool(np.all(np.diff(h0) >= 0))}"
)

step_count = np.searchsorted(ev_times, TIMES, side="right")
step_index = np.maximum(step_count - 1, 0)
h0_at = np.where(step_count > 0, h0[step_index], 0.0)
step_time = np.where(step_count > 0, ev_times[step_index], np.nan)
cox_H_hand = np.exp(cox_margin)[:, None] * h0_at
cox_H = COX._cum_hazard_at(cox_margin, TIMES)
peek("9.step_count", step_count)
peek("9.step_index", step_index)
peek("9.h0_at", h0_at)
peek("9.cox_H", cox_H)
same("Cox _cum_hazard_at", cox_H, cox_H_hand)

pd.DataFrame(
    {
        "t": TIMES,
        "моментов отказа <= t": step_count,
        "последний отказ <= t": step_time,
        "H0(t)": h0_at,
        "H(t)": cox_H[0],
    }
)

[9.ev_times] ndarray shape=(2543,) dtype=float64 size=2543 bytes=20344 | min=0.03 max=2347.64 nan=0 inf=0
    [3.00000e-02 9.00000e-02 1.50000e-01 ... 2.09355e+03 2.23751e+03 2.34764e+03]
[9.h0] ndarray shape=(2543,) dtype=float64 size=2543 bytes=20344 | min=0.000236034 max=6.2296 nan=0 inf=0
    [2.360344e-04 4.722465e-04 5.941000e-04 ... 5.291338e+00 5.704019e+00 6.229601e+00]
моменты отказов строго возрастают: True; H0 не убывает: True
[9.step_count] ndarray shape=(3,) dtype=int64 size=3 bytes=24 | min=411 max=1501
    [ 411 1104 1501]
[9.step_index] ndarray shape=(3,) dtype=int64 size=3 bytes=24 | min=410 max=1500
    [ 410 1103 1500]
[9.h0_at] ndarray shape=(3,) dtype=float64 size=3 bytes=24 | min=0.0692587 max=0.431829 nan=0 inf=0
    [0.069259 0.259815 0.431829]
[9.cox_H] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.370887 max=2.31249 nan=0 inf=0
    [[0.370887 1.391337 2.312485]]
[ok] Cox _cum_hazard_at: shape=(1, 3), max |разница| = 0.000e+00


,t,моментов отказа <= t,последний отказ <= t,H0(t),H(t)
0,50,411,49.89,0.0692587,0.370887
1,193.5,1104,193.5,0.259815,1.39134
2,293.5,1501,293.49,0.431829,2.31249


### 9.3 Связь хрупкости и выживаемость

Как в разделе 5: если у модели Cox есть `calibration`, перед $\exp(-H)$ накопленная опасность проходит замену $H \to \log(1 + v k H)/v$. Наклон связи $k/(1 + v k H)$ сохраняется для интенсивности на следующем шаге.

In [23]:
if COX.calibration is None:
    cox_H_used = cox_H
    cox_link_slope = np.ones_like(cox_H)
    print("calibration is None: H входит в exp(-H) без изменений")
else:
    cox_v = COX.calibration["frailty_variance"]
    cox_k = COX.calibration["scale"]
    if cox_v < 1e-8:
        cox_H_used = cox_k * cox_H
        cox_slope_hand = np.full_like(cox_H, cox_k)
    else:
        cox_H_used = np.log1p(cox_v * cox_k * cox_H) / cox_v
        cox_slope_hand = cox_k / (1.0 + cox_v * cox_k * cox_H)
    cox_H_linked, cox_link_slope = _frailty_link(cox_H, **COX.calibration)
    same("Cox: связь хрупкости", cox_H_used, cox_H_linked)
    same("Cox: наклон связи", cox_slope_hand, cox_link_slope)
    print(f"H заменено на log(1 + v*k*H) / v при v={cox_v:.4g}, k={cox_k:.4g}")

cox_S_hand = np.exp(-cox_H_used)
cox_S = COX.survival(drone, TIMES)
peek("9.cox_H_used", cox_H_used)
peek("9.cox_link_slope", cox_link_slope)
peek("9.cox_S", cox_S)
same("Cox _survival_at", COX._survival_at(cox_margin, TIMES), cox_S_hand)
same("Cox survival", cox_S, cox_S_hand)

pd.DataFrame(
    {
        "t": TIMES,
        "H": cox_H[0],
        "H после связи": cox_H_used[0],
        "наклон связи": cox_link_slope[0],
        "S": cox_S[0],
    }
)

[ok] Cox: связь хрупкости: shape=(1, 3), max |разница| = 2.220e-16
[ok] Cox: наклон связи: shape=(1, 3), max |разница| = 1.110e-16
H заменено на log(1 + v*k*H) / v при v=0.2678, k=1.045
[9.cox_H_used] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.368858 max=1.86391 nan=0 inf=0
    [[0.368858 1.228259 1.863911]]
[9.cox_link_slope] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.634522 max=0.946984 nan=0 inf=0
    [[0.946984 0.752284 0.634522]]
[9.cox_S] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.155065 max=0.691523 nan=0 inf=0
    [[0.691523 0.292802 0.155065]]
[ok] Cox _survival_at: shape=(1, 3), max |разница| = 2.776e-17
[ok] Cox survival: shape=(1, 3), max |разница| = 2.776e-17


,t,H,H после связи,наклон связи,S
0,50,0.370887,0.368858,0.946984,0.691523
1,193.5,1.39134,1.22826,0.752284,0.292802
2,293.5,2.31249,1.86391,0.634522,0.155065


### 9.4 Интенсивность, сглаженная ядром

$H_0$ — ступенчатая функция, поэтому её производная — набор всплесков в моментах отказов обучения. `hazard` размазывает каждый скачок $H_0$ ядром Епанечникова с полушириной `bandwidth` $b$, умножает на $\exp(\hat f)$ и на наклон связи:

$$h(t \mid x) = \frac{k}{1 + v k H(t \mid x)} \; e^{\hat f(x)} \sum_j K_b(t - t_j)\, \Delta H_0(t_j), \qquad K_b(d) = \frac{0.75}{b}\Big(1 - \frac{d^2}{b^2}\Big)\ \text{при}\ |d| < b.$$

Шейпы: скачки $\Delta H_0$ имеют шейп `(k,)`. Веса ядра образуют матрицу `(m, k)`: строка на момент оценки, столбец на момент отказа обучения; большинство элементов равны нулю, потому что вклад дают только моменты отказов в пределах одной ширины ядра от $t$. Матричное произведение даёт сглаженную базовую интенсивность, `(m,)`.

Ширина ядра — аргумент `hazard`, а не часть модели. В этом разделе она берётся по правилу Сильвермана на моментах отказов обучения. Дашборд берёт правило на тестовой таблице (раздел 11), поэтому его интенсивность Cox немного другая. В пределах одной ширины ядра от 0 и от последнего момента отказа обучения оценка смещена вниз: там часть окна ядра пуста.

In [24]:
cox_bandwidth = hazard_bandwidth(ev_times, np.ones(len(ev_times), dtype=bool))
h0_jumps = np.diff(h0, prepend=0.0)
kernel = _epanechnikov(TIMES, ev_times, cox_bandwidth)
kernel_support = (kernel > 0).sum(axis=1)
baseline_hazard = kernel @ h0_jumps
cox_h_unlinked = np.exp(cox_margin)[:, None] * baseline_hazard[None, :]
cox_h_hand = cox_link_slope * cox_h_unlinked
cox_h = COX.hazard(drone, TIMES, bandwidth=cox_bandwidth)
peek("9.cox_bandwidth", cox_bandwidth)
peek("9.h0_jumps", h0_jumps)
peek("9.kernel", kernel)
peek("9.kernel_support", kernel_support)
peek("9.baseline_hazard", baseline_hazard)
peek("9.cox_h_unlinked", cox_h_unlinked)
peek("9.cox_h", cox_h)
same("Cox hazard", cox_h, cox_h_hand)

pd.DataFrame(
    {
        "t": TIMES,
        "моментов отказа в окне ядра": kernel_support,
        "базовая интенсивность": baseline_hazard,
        "наклон связи": cox_link_slope[0],
        "h, на час": cox_h[0],
        f"h, на {HAZARD_SCALE:g} ч": HAZARD_SCALE * cox_h[0],
    }
)

[9.cox_bandwidth] float = 49.371924407013864
[9.h0_jumps] ndarray shape=(2543,) dtype=float64 size=2543 bytes=20344 | min=0.000121854 max=0.525582 nan=0 inf=0
    [2.360344e-04 2.362121e-04 1.218535e-04 ... 2.597865e-01 4.126814e-01 5.255822e-01]
[9.kernel] ndarray shape=(3, 2543) dtype=float64 size=7629 bytes=61032 | min=0 max=0.0151908 nan=0 inf=0
    [[0. 0. 0. ... 0. 0. 0.]
     [0. 0. 0. ... 0. 0. 0.]
     [0. 0. 0. ... 0. 0. 0.]]
[9.kernel_support] ndarray shape=(3,) dtype=int64 size=3 bytes=24 | min=323 max=675
    [675 409 323]
[9.baseline_hazard] ndarray shape=(3,) dtype=float64 size=3 bytes=24 | min=0.00128666 max=0.00170974 nan=0 inf=0
    [0.001287 0.00146  0.00171 ]
[9.cox_h_unlinked] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.00689021 max=0.00915583 nan=0 inf=0
    [[0.00689  0.007821 0.009156]]
[9.cox_h] ndarray shape=(1, 3) dtype=float64 size=3 bytes=24 | min=0.00580958 max=0.00652492 nan=0 inf=0
    [[0.006525 0.005884 0.00581 ]]
[ok] Cox hazard: shape=

,t,моментов отказа в окне ядра,базовая интенсивность,наклон связи,"h, на час","h, на 100 ч"
0,50,675,0.00128666,0.946984,0.00652492,0.652492
1,193.5,409,0.00146045,0.752284,0.00588351,0.588351
2,293.5,323,0.00170974,0.634522,0.00580958,0.580958


### 9.5 Условная выживаемость и конец базовой линии

Отношение $S(a+u)/S(a)$ считается как в разделе 6. Одна ловушка есть только у Cox: $H_0$ плоская после последнего момента отказа обучения, поэтому дальше модель не предсказывает новых отказов и условная кривая слишком оптимистична. `conditional_survival` пишет в лог предупреждение, когда $a + u$ уходит за этот момент. Вызов с $u = 0$ и $u = 100$ остаётся внутри базовой линии; отдельный вызов на час дальше последнего отказа обучения предупреждение вызывает.

Последняя таблица ставит две модели рядом в контрольные моменты.

In [25]:
cox_s_age = COX._survival_at(cox_margin, ages_col)
cox_s_ahead = COX._survival_at(cox_margin, future)
cox_cond = COX.conditional_survival(drone, ages, U)
peek("9.cox_s_age", cox_s_age)
peek("9.cox_s_ahead", cox_s_ahead)
peek("9.cox_cond", cox_cond)
same("Cox conditional_survival", cox_cond, cox_s_ahead / cox_s_age)

last_train_failure = float(ev_times[-1])
past_u = np.array([last_train_failure - AGE + 1.0])
print(
    f"\nпоследний отказ обучения в {last_train_failure:.1f} ч; "
    f"вызов с u={past_u[0]:.1f} ч уходит за базовую линию:"
)
cox_cond_past = COX.conditional_survival(drone, ages, past_u)
peek("9.cox_cond_past", cox_cond_past)

display(
    pd.DataFrame(
        {
            "u": U,
            "S(a)": np.full(len(U), cox_s_age[0, 0]),
            "S(a+u)": cox_s_ahead[0],
            "S(a+u)/S(a)": cox_cond[0],
        }
    )
)
pd.DataFrame(
    {
        "t": TIMES,
        "S AFT": S[0],
        "S Cox": cox_S[0],
        f"h AFT, на {HAZARD_SCALE:g} ч": HAZARD_SCALE * h[0],
        f"h Cox, на {HAZARD_SCALE:g} ч": HAZARD_SCALE * cox_h[0],
    }
)

[9.cox_s_age] ndarray shape=(1, 1) dtype=float64 size=1 bytes=8 | min=0.292802 max=0.292802 nan=0 inf=0
    [[0.292802]]
[9.cox_s_ahead] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=0.155065 max=0.292802 nan=0 inf=0
    [[0.292802 0.155065]]
[9.cox_cond] ndarray shape=(1, 2) dtype=float64 size=2 bytes=16 | min=0.52959 max=1 nan=0 inf=0
    [[1.      0.52959]]
[ok] Cox conditional_survival: shape=(1, 2), max |разница| = 0.000e+00

последний отказ обучения в 2347.6 ч; вызов с u=2155.1 ч уходит за базовую линию:
Cox baseline hazard is flat after 2348 h; conditional survival beyond it is optimistic


[9.cox_cond_past] ndarray shape=(1, 1) dtype=float64 size=1 bytes=8 | min=0.00055653 max=0.00055653 nan=0 inf=0
    [[0.000557]]


,u,S(a),S(a+u),S(a+u)/S(a)
0,0,0.292802,0.292802,1
1,100,0.292802,0.155065,0.52959


,t,S AFT,S Cox,"h AFT, на 100 ч","h Cox, на 100 ч"
0,50,0.709194,0.691523,0.787901,0.652492
1,193.5,0.280545,0.292802,0.503687,0.588351
2,293.5,0.181514,0.155065,0.377782,0.580958


## 10. Вся тестовая таблица

Всё выше считалось при `n = 1`. Дашборд и метрики работают с `artifacts/predictions.parquet`: это тестовая выборка с предсказаниями, которые добавил `drone_survival.ipynb` (`risk_cox`, `risk_aft` оценочных моделей; `forecast_risk_cox`, `forecast_risk_aft` боевых моделей; `risk_group`, столбцы остаточного ресурса и `priority` боевой модели, названной в `forecast_name`).

### 10.1 Сохранённые предсказания против моделей на диске

Те же методы теперь возвращают строку на дрон: `risk` имеет шейп `(n,)`, `survival` — `(n, m)`. Две сверки:

- сохранённые столбцы `forecast_risk_*` равны `Model.risk` по файлам боевых моделей, а столбцы `risk_*` — по файлам оценочных, то есть таблица и модели получены в одном прогоне;
- если прослеживаемый дрон есть в тестовой таблице, его строка пакетного результата равна предсказанию для одной строки из разделов выше, то есть пакетный расчёт ничего не меняет.

Моменты времени — `eval_times` из `meta.json`, сетка, на которой считались метрики.

In [26]:
pred = pd.read_parquet(ARTIFACTS_DIR / "predictions.parquet")
MODELS = {"cox": COX, "aft": MODEL}
EVAL_MODELS = {kind: Model.load(MODELS_DIR / name) for kind, name in META["models"].items()}
EVAL_TIMES = np.array(META["eval_times"]["test"])
peek("10.pred", pred)
peek("10.EVAL_TIMES", EVAL_TIMES)

drone_positions = np.flatnonzero(pred["drone_id"].to_numpy() == DRONE_ID)
IN_TEST = len(drone_positions) == 1
DRONE_POS = int(drone_positions[0]) if IN_TEST else None
print(f"{DRONE_ID} в тестовой таблице: {IN_TEST}" + (f", строка {DRONE_POS}" if IN_TEST else ""))

BATCH_RISK = {kind: model.risk(pred) for kind, model in MODELS.items()}
BATCH_SURV = {kind: model.survival(pred, EVAL_TIMES) for kind, model in MODELS.items()}
for kind, model in MODELS.items():
    peek(f"10.BATCH_RISK.{kind}", BATCH_RISK[kind])
    peek(f"10.BATCH_SURV.{kind}", BATCH_SURV[kind])
    same(
        f"{kind}: сохранённый forecast_risk_{kind} = Model.risk боевой модели",
        pred[f"forecast_risk_{kind}"],
        BATCH_RISK[kind],
    )
    same(
        f"{kind}: сохранённый risk_{kind} = Model.risk оценочной модели",
        pred[f"risk_{kind}"],
        EVAL_MODELS[kind].risk(pred),
    )
    if IN_TEST:
        row = slice(DRONE_POS, DRONE_POS + 1)
        same(f"{kind}: строка {DRONE_POS} пакета = risk одной строки", BATCH_RISK[kind][row], model.risk(drone))
        same(
            f"{kind}: строка {DRONE_POS} пакета = survival одной строки",
            BATCH_SURV[kind][row],
            model.survival(drone, EVAL_TIMES),
        )

pred.loc[pred["drone_id"] == DRONE_ID].T

[10.pred] DataFrame shape=(1001, 33) dtype=16 x float64, 6 x str, 4 x int64, 4 x float32, 2 x bool, 1 x datetime64[us] size=33033 bytes=281024
    столбцы: drone_id, model, environment, commission_date, is_refurbished, initial_battery_cycles,
    firmware_major, maintenance_interval_h, operator_experience_yrs, daily_utilization_h,
    avg_flight_min, avg_payload_kg, payload_ratio, mean_temp_c, avg_humidity_pct, avg_wind_ms,
    hard_landings_per_100h, mean_vibration_g, duration_h, event, failure_mode, in_service, risk_cox,
    risk_aft, forecast_risk_cox, forecast_risk_aft, risk_group, age_h, rul_q10, rul_median, rmrl,
    p_fail_window, priority
[10.EVAL_TIMES] ndarray shape=(50,) dtype=float64 size=50 bytes=400 | min=21.4565 max=385.48 nan=0 inf=0
    [ 21.4565    28.885551  36.314602 ... 370.621898 378.050949 385.48    ]
DR-003057 в тестовой таблице: True, строка 613


[10.BATCH_RISK.cox] ndarray shape=(1001,) dtype=float32 size=1001 bytes=4004 | min=-1.38264 max=4.96308 nan=0 inf=0
    [-0.759229 -0.459004 -0.074707 ...  0.196598 -0.900121  0.878095]
[10.BATCH_SURV.cox] ndarray shape=(1001, 50) dtype=float64 size=50050 bytes=400400 | min=6.43041e-06 max=0.992524 nan=0 inf=0
    [[0.986112 0.981603 0.977164 ... 0.767834 0.763061 0.757715]
     [0.981307 0.975263 0.969325 ... 0.702971 0.697216 0.690787]
     [0.972701 0.963939 0.955362 ... 0.602422 0.595487 0.587776]
     ...
     [0.964386 0.953037 0.941967 ... 0.521182 0.513627 0.505259]
     [0.987923 0.983996 0.980127 ... 0.794127 0.789798 0.784945]
     [0.931135 0.90983  0.889341 ... 0.302903 0.295446 0.287301]]
[ok] cox: сохранённый forecast_risk_cox = Model.risk боевой модели: shape=(1001,), max |разница| = 0.000e+00
[ok] cox: сохранённый risk_cox = Model.risk оценочной модели: shape=(1001,), max |разница| = 0.000e+00
[ok] cox: строка 613 пакета = risk одной строки: shape=(1,), max |разница| =

,613
drone_id,DR-003057
model,quad_light
environment,urban
commission_date,2025-04-17 00:00:00
is_refurbished,False
initial_battery_cycles,24
firmware_major,3
maintenance_interval_h,200
operator_experience_yrs,1.44
daily_utilization_h,0.75


### 10.2 Что получают метрики

`utils.evaluate` передаёт в `sksurv` три массива:

- `_surv(data)`: наблюдаемые записи `(event, time)`, шейп `(n,)`;
- `model.survival(data, times)`: предсказанная $S(t)$ на сетке оценки, шейп `(n, m)`; Brier score использует её напрямую, а AUC — в виде $1 - S$;
- `model.risk(data)`: ранжирующий скор, шейп `(n,)`; его используют оба C-index.

Распределение цензурирования для весов IPCW оценивается на самих оцениваемых строках, как в `drone_survival.ipynb`. Опубликованные метрики описывают **оценочные** модели, поэтому ячейка пересчитывает тестовые метрики по ним и сверяет результат с `artifacts/metrics.parquet`. Если они совпали, модели, таблица предсказаний и опубликованные метрики согласованы между собой. `brier` и `auc` — массивы по моментам времени с шейпом `(m,)`, кривые графиков качества модели.

In [27]:
y_test = _surv(pred)
peek("10.y_test", y_test)
print(f"отказов: {int(y_test['event'].sum())} из {len(y_test)} записей")

stored_metrics = pd.read_parquet(ARTIFACTS_DIR / "metrics.parquet")
peek("10.stored_metrics", stored_metrics)

EVALUATION = {
    kind: evaluate(model, pred, pred, EVAL_TIMES) for kind, model in EVAL_MODELS.items()
}
for kind, result in EVALUATION.items():
    peek(f"10.EVALUATION.{kind}.brier", result["brier"])
    peek(f"10.EVALUATION.{kind}.auc", result["auc"])
    on_test = (stored_metrics["model"] == kind) & (stored_metrics["split"] == "test")
    stored = stored_metrics[on_test].iloc[0]
    for metric in SCALAR_METRICS:
        same(
            f"{kind} {metric} = metrics.parquet",
            np.array([result[metric]]),
            np.array([stored[metric]]),
            rtol=1e-9,
        )

pd.DataFrame(
    {
        kind: {metric: result[metric] for metric in SCALAR_METRICS}
        for kind, result in EVALUATION.items()
    }
).T

[10.y_test] ndarray shape=(1001,) dtype=[('event', '?'), ('time', '<f8')] size=1001 bytes=9009
    [( True,  63.67) ( True, 159.67) (False, 312.4 ) ... ( True,  66.78) (False, 227.58)
     ( True,   1.03)]
отказов: 448 из 1001 записей
[10.stored_metrics] DataFrame shape=(4, 6) dtype=4 x float64, 2 x str size=24 bytes=350
    столбцы: model, split, c_index, c_index_ipcw, mean_auc, ibs


[10.EVALUATION.cox.brier] ndarray shape=(50,) dtype=float64 size=50 bytes=400 | min=0.0484388 max=0.221214 nan=0 inf=0
    [0.048439 0.057875 0.068015 ... 0.219005 0.218536 0.220104]
[10.EVALUATION.cox.auc] ndarray shape=(50,) dtype=float64 size=50 bytes=400 | min=0.67903 max=0.713779 nan=0 inf=0
    [0.70626  0.712709 0.687843 ... 0.710284 0.713779 0.709326]
[ok] cox c_index = metrics.parquet: shape=(1,), max |разница| = 0.000e+00
[ok] cox c_index_ipcw = metrics.parquet: shape=(1,), max |разница| = 0.000e+00
[ok] cox mean_auc = metrics.parquet: shape=(1,), max |разница| = 0.000e+00
[ok] cox ibs = metrics.parquet: shape=(1,), max |разница| = 0.000e+00
[10.EVALUATION.aft.brier] ndarray shape=(50,) dtype=float64 size=50 bytes=400 | min=0.0498959 max=0.227226 nan=0 inf=0
    [0.049896 0.059202 0.069606 ... 0.224129 0.223605 0.225016]
[10.EVALUATION.aft.auc] ndarray shape=(50,) dtype=float64 size=50 bytes=400 | min=0.674577 max=0.699807 nan=0 inf=0
    [0.689536 0.699748 0.674577 ... 0.689

,c_index,c_index_ipcw,mean_auc,ibs
cox,0.671746,0.664759,0.699508,0.170899
aft,0.665767,0.658315,0.687146,0.175398


## 11. Настройки и сетки дашборда

### 11.1 Настройки и ширина ядра

Дашборд не рисует кривые в трёх контрольных моментах. `Settings.from_data` выводит всё, что нужно графику, из `meta.json` и тестовых предсказаний: `horizon` и `window` остаточного ресурса, пороги приоритета, три сетки времени и ширину ядра.

`hazard_bandwidth` — правило Сильвермана на моментах отказов тестовой таблицы:

$$b = 0.9 \cdot \min(\mathrm{sd},\ \mathrm{IQR}/1.34) \cdot n^{-1/5},$$

где $n$ — число отказов. Ячейка пересчитывает её вручную.

In [28]:
DATA = DashboardData(
    meta=META, tables={"predictions": pred}, models=EVAL_MODELS, production=MODELS, card=CARD
)
SETTINGS = Settings.from_data(DATA)
FAILED = EVENT == 1
settings_fields = {
    field.name: getattr(SETTINGS, field.name) for field in dataclasses.fields(SETTINGS)
}
peek("11.SETTINGS", settings_fields, limit=len(settings_fields))

event_durations = pred.loc[pred["event"] == 1, "duration_h"].to_numpy(dtype=float)
q75, q25 = np.quantile(event_durations, [0.75, 0.25])
iqr = q75 - q25
sd = event_durations.std(ddof=1)
spread = min(sd, iqr / 1.34) if iqr > 0 else sd
bandwidth_hand = 0.9 * spread * len(event_durations) ** -0.2
peek("11.event_durations", event_durations)
same(
    "hazard_bandwidth = правило Сильвермана",
    np.array([SETTINGS.hazard_bandwidth]),
    np.array([bandwidth_hand]),
)
print(
    f"sd={sd:.2f} ч, IQR/1.34={iqr / 1.34:.2f} ч, n={len(event_durations)} отказов "
    f"-> ширина ядра {bandwidth_hand:.2f} ч"
)
print(
    f"возраст {AGE:g} ч; окно {SETTINGS.window:g} ч; горизонт остаточного ресурса {SETTINGS.horizon:g} ч; "
    f"отказ наблюдался: {FAILED}"
)

[11.SETTINGS] dict size=12
    'best': 'aft'
    'forecast': 'cox'
    'features': list size=16
    'cat_features': ['model', 'environment', 'is_refurbished', 'firmware_major']
    'horizon': 1111.0
    'window': 100.0
    'p_high': 0.5
    'p_medium': 0.2
    'surv_grid': ndarray shape=(300,) dtype=float64 size=300 bytes=2400
    'rul_grid': ndarray shape=(300,) dtype=float64 size=300 bytes=2400
    'hazard_bandwidth': 42.439288934251536
    'hazard_grid': ndarray shape=(200,) dtype=float64 size=200 bytes=1600
[11.event_durations] ndarray shape=(448,) dtype=float64 size=448 bytes=3584 | min=0.55 max=1156.68 nan=0 inf=0
    [ 63.67 159.67 130.12 ...  41.49  66.78   1.03]
[ok] hazard_bandwidth = правило Сильвермана: shape=(1,), max |разница| = 0.000e+00
sd=159.88 ч, IQR/1.34=164.95 ч, n=448 отказов -> ширина ядра 42.44 ч
возраст 193.5 ч; окно 100 ч; горизонт остаточного ресурса 1111 ч; отказ наблюдался: False


### 11.2 Сетки

- `surv_grid`, кривые выживаемости: 300 точек от 0 до наибольшего `duration_h` в тестовой таблице.
- `rul_grid`, условные кривые: 300 точек от 0 до `rul_horizon` из `meta.json`. Её единица — дополнительные лётные часы, а не общий налёт (раздел 14).
- `hazard_grid`, интенсивность флота: 200 точек от одной ширины ядра до 90-го перцентиля `duration_h`. У нуля сглаженная оценка смещена, а в хвосте опирается на слишком малое число дронов, поэтому сетка туда не заходит.
- `drone_hazard_grid(settings, age)`, интенсивность одного дрона: 300 точек с тем же левым краем; правый край сдвигается до `age + window`, когда это дальше 90-го перцентиля. Дальше кривую не тянут: там базовая линия Cox опирается на немногие отказы обучения.

Возраст обычно попадает между двумя узлами сетки. Вертикаль графика рисуется в точном `AGE`, а кривая — ломаная по узлам. Таблица даёт для каждой сетки в часах общего налёта ближайший к возрасту узел и его сдвиг.

In [29]:
same("surv_grid", SETTINGS.surv_grid, np.linspace(0.0, pred["duration_h"].max(), 300))
same("rul_grid", SETTINGS.rul_grid, np.linspace(0.0, META["rul_horizon"], 300))
same(
    "hazard_grid",
    SETTINGS.hazard_grid,
    np.linspace(SETTINGS.hazard_bandwidth, pred["duration_h"].quantile(0.9), 200),
)
hazard_end = max(float(SETTINGS.hazard_grid[-1]), AGE + SETTINGS.window)
DRONE_HAZARD_GRID = drone_hazard_grid(SETTINGS, AGE)
same(
    "drone_hazard_grid",
    DRONE_HAZARD_GRID,
    np.linspace(SETTINGS.hazard_bandwidth, hazard_end, 300),
)
peek("11.surv_grid", SETTINGS.surv_grid)
peek("11.rul_grid", SETTINGS.rul_grid)
peek("11.hazard_grid", SETTINGS.hazard_grid)
peek("11.DRONE_HAZARD_GRID", DRONE_HAZARD_GRID)


def nearest(grid: np.ndarray) -> tuple[int, float]:
    """Индекс и значение узла ``grid``, ближайшего к ``AGE``."""
    i = int(np.argmin(np.abs(grid - AGE)))
    return i, float(grid[i])


grid_rows = []
for name, grid in (
    ("surv_grid", SETTINGS.surv_grid),
    ("hazard_grid", SETTINGS.hazard_grid),
    ("drone_hazard_grid", DRONE_HAZARD_GRID),
):
    _, node = nearest(grid)
    grid_rows.append(
        {
            "сетка": name,
            "точек": len(grid),
            "от": float(grid[0]),
            "до": float(grid[-1]),
            "шаг": float(grid[1] - grid[0]),
            "узел рядом с возрастом": node,
            "сдвиг, ч": node - AGE,
        }
    )
print(
    f"\nправый край сетки интенсивности дрона = max(90-й перцентиль "
    f"{SETTINGS.hazard_grid[-1]:.1f}, возраст + окно {AGE + SETTINGS.window:.1f}) = {hazard_end:.1f} ч"
)
pd.DataFrame(grid_rows)

[ok] surv_grid: shape=(300,), max |разница| = 0.000e+00
[ok] rul_grid: shape=(300,), max |разница| = 0.000e+00
[ok] hazard_grid: shape=(200,), max |разница| = 0.000e+00
[ok] drone_hazard_grid: shape=(300,), max |разница| = 0.000e+00
[11.surv_grid] ndarray shape=(300,) dtype=float64 size=300 bytes=2400 | min=0 max=1236.24 nan=0 inf=0
    [   0.          4.134582    8.269164 ... 1227.970836 1232.105418 1236.24    ]
[11.rul_grid] ndarray shape=(300,) dtype=float64 size=300 bytes=2400 | min=0 max=1111 nan=0 inf=0
    [   0.          3.715719    7.431438 ... 1103.568562 1107.284281 1111.      ]
[11.hazard_grid] ndarray shape=(200,) dtype=float64 size=200 bytes=1600 | min=42.4393 max=495.71 nan=0 inf=0
    [ 42.439289  44.717031  46.994773 ... 491.154515 493.432258 495.71    ]
[11.DRONE_HAZARD_GRID] ndarray shape=(300,) dtype=float64 size=300 bytes=2400 | min=42.4393 max=495.71 nan=0 inf=0
    [ 42.439289  43.955244  45.4712   ... 492.678089 494.194044 495.71    ]

правый край сетки интенсив

,сетка,точек,от,до,шаг,узел рядом с возрастом,"сдвиг, ч"
0,surv_grid,300,0,1236.24,4.13458,194.325,0.825351
1,hazard_grid,200,42.4393,495.71,2.27774,192.77,-0.729721
2,drone_hazard_grid,300,42.4393,495.71,1.51596,194.035,0.534844


## 12. Кривая выживаемости одного дрона

`figures.drone_survival` добавляет по одной линии на модель с `x = surv_grid` и `y = model.survival(row, surv_grid)[0]`: `survival` возвращает `(1, 300)` для единственной строки, а `[0]` берёт кривую `(300,)`. Ось y — $S(t)$ с фиксированным диапазоном.

Вертикаль отмечает `duration_h`: зелёная с подписью «flown» при `event = 0`, красная с подписью «failed at», если отказ уже наблюдался. Это фигура разметки (`layout.shapes`), а не след. Её x — значение данных; её y идёт от 0 до 1 в долях оси (`y domain`), поэтому она всегда занимает всю высоту.

Карточка дрона в дашборде делает тот же вызов: `drone_survival(FORECAST_MODELS, rows, SETTINGS.surv_grid, row["duration_h"], failed)`, где `FORECAST_MODELS` — боевые модели, здесь `MODELS`.

Ячейка строит график, перечисляет его следы и разметку, пересчитывает каждую кривую и сверяет её с тем, что лежит в следе.

In [30]:
surv_fig = drone_survival(MODELS, drone, SETTINGS.surv_grid, AGE, FAILED)
describe_figure("12.surv_fig", surv_fig)

SURV_CURVES = {kind: model.survival(drone, SETTINGS.surv_grid) for kind, model in MODELS.items()}
for kind, curve in SURV_CURVES.items():
    peek(f"12.SURV_CURVES.{kind}", curve)
    drawn = trace_xy(surv_fig, MODEL_LABELS[kind])
    same(f"{kind}: x следа = surv_grid", drawn["x"], SETTINGS.surv_grid)
    same(f"{kind}: y следа = survival(surv_grid)[0]", drawn["y"], curve[0])

surv_vline = surv_fig.layout.shapes[0]
same(
    "вертикаль в AGE",
    np.array([surv_vline.x0, surv_vline.x1], dtype=float),
    np.array([AGE, AGE]),
)
print("подпись вертикали:", surv_fig.layout.annotations[0].text)

[12.surv_fig] Figure: следов 2, shapes 1, annotations 1
    заголовок: 'Probability of operating without failure'
    ось x: 'Flight hours', range=None, rangemode=None
    ось y: 'S(t)', range=(0, 1.02), rangemode=None
    высота: 400


,след,имя,тип,режим,шейп x,шейп y,x min,x max,y min,y max,NaN в y,цвет линии,толщина линии,заливка,в легенде
0,0,Cox,scatter,lines,"(300,)","(300,)",0,1236.24,0.00289175,1,0,#1f77b4,None,None,True
1,1,AFT,scatter,lines,"(300,)","(300,)",0,1236.24,0.0306471,1,0,#ff7f0e,None,None,True


,вид,xref,x0,x1,yref,y0,y1,цвет,штрих,текст
0,shape/line,x,193.5,193.5,y domain,0,1,green,dash,NaN
1,annotation,x,193.5,193.5,y domain,1,1,NaN,NaN,flown 194 h


[12.SURV_CURVES.cox] ndarray shape=(1, 300) dtype=float64 size=300 bytes=2400 | min=0.00289175 max=1 nan=0 inf=0
    [[1.       0.968987 0.940087 ... 0.002999 0.002892 0.002892]]
[ok] cox: x следа = surv_grid: shape=(300,), max |разница| = 0.000e+00
[ok] cox: y следа = survival(surv_grid)[0]: shape=(300,), max |разница| = 0.000e+00
[12.SURV_CURVES.aft] ndarray shape=(1, 300) dtype=float64 size=300 bytes=2400 | min=0.0306471 max=1 nan=0 inf=0
    [[1.       0.986187 0.965417 ... 0.030918 0.030782 0.030647]]
[ok] aft: x следа = surv_grid: shape=(300,), max |разница| = 0.000e+00
[ok] aft: y следа = survival(surv_grid)[0]: shape=(300,), max |разница| = 0.000e+00
[ok] вертикаль в AGE: shape=(2,), max |разница| = 0.000e+00
подпись вертикали: flown 194 h


$S$ в узле сетки — вершина ломаной. $S(\mathrm{AGE})$ из раздела 8 лежит на вертикали и совпадает с вершиной, только если возраст сам является узлом `surv_grid`. Ячейка показывает график, значения на графике в узлах вокруг возраста и точные значения в контрольные моменты.

In [31]:
surv_plot_table = pd.DataFrame(
    {"t": SETTINGS.surv_grid, **{f"S {kind}": curve[0] for kind, curve in SURV_CURVES.items()}}
)
peek("12.surv_plot_table", surv_plot_table)

surv_node_i, surv_node_t = nearest(SETTINGS.surv_grid)
print(
    f"узел surv_grid рядом с возрастом: {surv_node_t:.4f} ч, "
    f"S(AFT) = {SURV_CURVES['aft'][0, surv_node_i]:.6f}; точное S({AGE:g}) = {s_now:.6f}"
)
display(surv_fig)
print("значения на графике в узлах вокруг возраста:")
display(surv_plot_table.iloc[max(surv_node_i - 2, 0) : surv_node_i + 3])
print("точные значения в контрольные моменты:")
pd.DataFrame(
    {"t": TIMES, **{f"S {kind}": model.survival(drone, TIMES)[0] for kind, model in MODELS.items()}}
)

[12.surv_plot_table] DataFrame shape=(300, 3) dtype=3 x float64 size=900 bytes=7332
    столбцы: t, S cox, S aft
узел surv_grid рядом с возрастом: 194.3254 ч, S(AFT) = 0.279383; точное S(193.5) = 0.280545


значения на графике в узлах вокруг возраста:


,t,S cox,S aft
45,186.056,0.304167,0.291395
46,190.191,0.2974,0.285286
47,194.325,0.291146,0.279383
48,198.46,0.282562,0.273677
49,202.595,0.274938,0.26816


точные значения в контрольные моменты:


,t,S cox,S aft
0,50,0.691523,0.709194
1,193.5,0.292802,0.280545
2,293.5,0.155065,0.181514


## 13. Кривая интенсивности одного дрона

`figures.drone_hazard` рисует не саму $h(t)$, а `HAZARD_SCALE * model.hazard(row, grid, bandwidth)[0]` при `HAZARD_SCALE = 100`. Подпись оси «Failures per 100 flight hours» — скорость отказов на 100 лётных часов.

Значения x — `drone_hazard_grid(SETTINGS, age)`. Линия Cox использует `SETTINGS.hazard_bandwidth`, поэтому она отличается от интенсивности Cox из раздела 9.4, посчитанной с другой шириной ядра. Вертикаль — фигура разметки того же вида, что в разделе 12.

Левый край сетки — одна ширина ядра, а не ноль. У контрольного момента левее него есть значение интенсивности в таблице скоростей ниже, но нет точки на линии.

In [32]:
hazard_fig = drone_hazard(
    MODELS, drone, DRONE_HAZARD_GRID, SETTINGS.hazard_bandwidth, AGE, FAILED
)
describe_figure("13.hazard_fig", hazard_fig)

HAZARD_CURVES = {
    kind: model.hazard(drone, DRONE_HAZARD_GRID, SETTINGS.hazard_bandwidth)
    for kind, model in MODELS.items()
}
for kind, curve in HAZARD_CURVES.items():
    peek(f"13.HAZARD_CURVES.{kind}", curve)
    drawn = trace_xy(hazard_fig, MODEL_LABELS[kind])
    same(f"{kind}: x следа = drone_hazard_grid", drawn["x"], DRONE_HAZARD_GRID)
    same(f"{kind}: y следа = {HAZARD_SCALE:g} * hazard", drawn["y"], HAZARD_SCALE * curve[0])

hazard_vline = hazard_fig.layout.shapes[0]
same(
    "вертикаль в AGE",
    np.array([hazard_vline.x0, hazard_vline.x1], dtype=float),
    np.array([AGE, AGE]),
)
display(hazard_fig)

[13.hazard_fig] Figure: следов 2, shapes 1, annotations 1
    заголовок: 'Failure rate of this drone by flight hours'
    ось x: 'Flight hours', range=None, rangemode=None
    ось y: 'Failures per 100 flight hours', range=None, rangemode=tozero
    высота: 400


,след,имя,тип,режим,шейп x,шейп y,x min,x max,y min,y max,NaN в y,цвет линии,толщина линии,заливка,в легенде
0,0,Cox,scatter,lines,"(300,)","(300,)",42.4393,495.71,0.448933,0.668797,0,#1f77b4,None,None,True
1,1,AFT,scatter,lines,"(300,)","(300,)",42.4393,495.71,0.246415,0.789521,0,#ff7f0e,None,None,True


,вид,xref,x0,x1,yref,y0,y1,цвет,штрих,текст
0,shape/line,x,193.5,193.5,y domain,0,1,green,dash,NaN
1,annotation,x,193.5,193.5,y domain,1,1,NaN,NaN,flown 194 h


[13.HAZARD_CURVES.cox] ndarray shape=(1, 300) dtype=float64 size=300 bytes=2400 | min=0.00448933 max=0.00668797 nan=0 inf=0
    [[0.006594 0.006558 0.006522 ... 0.004489 0.004505 0.004517]]
[ok] cox: x следа = drone_hazard_grid: shape=(300,), max |разница| = 0.000e+00
[ok] cox: y следа = 100 * hazard: shape=(300,), max |разница| = 0.000e+00
[13.HAZARD_CURVES.aft] ndarray shape=(1, 300) dtype=float64 size=300 bytes=2400 | min=0.00246415 max=0.00789521 nan=0 inf=0
    [[0.007891 0.007895 0.007895 ... 0.002477 0.002471 0.002464]]
[ok] aft: x следа = drone_hazard_grid: shape=(300,), max |разница| = 0.000e+00
[ok] aft: y следа = 100 * hazard: shape=(300,), max |разница| = 0.000e+00
[ok] вертикаль в AGE: shape=(2,), max |разница| = 0.000e+00


**Скорость сейчас против вероятности за окно.** Высота линии в `AGE` — это скорость, а не вероятность скорого отказа. Вероятность отказа за следующие `window` часов — величина из раздела 6:

$$1 - S(a + w \mid x) \,/\, S(a \mid x).$$

Если бы скорость не менялась, вероятность была бы $1 - e^{-h(a)\,w}$. На протяжении окна кривая растёт или падает, поэтому одна только высота в `AGE` эту вероятность не даёт. Ячейка печатает оба числа для каждой модели и помечает, какие из контрольных моментов лежат внутри отрезка, покрытого графиком.

In [33]:
window = float(SETTINGS.window)
window_arr = np.array([window])
in_range = (TIMES >= DRONE_HAZARD_GRID[0]) & (TIMES <= DRONE_HAZARD_GRID[-1])
rate_columns = {"t": TIMES, "внутри отрезка графика": in_range}
window_rows = []
for kind, model in MODELS.items():
    rate = model.hazard(drone, TIMES, SETTINGS.hazard_bandwidth)[0]
    rate_columns[f"h {kind}, на час"] = rate
    rate_columns[f"h {kind}, на {HAZARD_SCALE:g} ч"] = HAZARD_SCALE * rate
    s_at_age = model.survival(drone, ages)[0, 0]
    s_at_window = model.survival(drone, ages + window)[0, 0]
    p_fail = 1.0 - s_at_window / s_at_age
    p_fail_cond = 1.0 - model.conditional_survival(drone, ages, window_arr)[0, 0]
    same(f"{kind}: 1 - S(a+w)/S(a)", np.array([p_fail]), np.array([p_fail_cond]))
    rate_now = float(model.hazard(drone, ages, SETTINGS.hazard_bandwidth)[0, 0])
    window_rows.append(
        {
            "модель": kind,
            f"скорость в текущем возрасте, на {HAZARD_SCALE:g} ч": HAZARD_SCALE * rate_now,
            f"P(отказ за {window:g} ч)": p_fail,
            "то же при неизменной скорости": 1.0 - np.exp(-rate_now * window),
        }
    )

window_table = pd.DataFrame(window_rows)
rate_table = pd.DataFrame(rate_columns)
peek("13.window_table", window_table)
peek("13.rate_table", rate_table)
outside = TIMES[~in_range]
print(
    f"\nлиния интенсивности покрывает {DRONE_HAZARD_GRID[0]:.1f}–{DRONE_HAZARD_GRID[-1]:.1f} ч; "
    + (
        f"контрольные моменты вне её: {outside.tolist()} ч"
        if len(outside)
        else "все контрольные моменты лежат внутри"
    )
)
display(window_table)
rate_table

[ok] cox: 1 - S(a+w)/S(a): shape=(1,), max |разница| = 0.000e+00
[ok] aft: 1 - S(a+w)/S(a): shape=(1,), max |разница| = 0.000e+00
[13.window_table] DataFrame shape=(2, 4) dtype=3 x float64, 1 x str size=8 bytes=202
    столбцы: модель, скорость в текущем возрасте, на 100 ч, P(отказ за 100 ч), то же при неизменной
    скорости
[13.rate_table] DataFrame shape=(3, 6) dtype=5 x float64, 1 x bool size=18 bytes=255
    столбцы: t, внутри отрезка графика, h cox, на час, h cox, на 100 ч, h aft, на час, h aft, на 100 ч

линия интенсивности покрывает 42.4–495.7 ч; все контрольные моменты лежат внутри


,модель,"скорость в текущем возрасте, на 100 ч",P(отказ за 100 ч),то же при неизменной скорости
0,cox,0.582358,0.47041,0.44142
1,aft,0.503687,0.352994,0.395701


,t,внутри отрезка графика,"h cox, на час","h cox, на 100 ч","h aft, на час","h aft, на 100 ч"
0,50,True,0.00644312,0.644312,0.00787901,0.787901
1,193.5,True,0.00582358,0.582358,0.00503687,0.503687
2,293.5,True,0.00589016,0.589016,0.00377782,0.377782


## 14. Условная кривая и остаточный ресурс

### 14.1 Как `remaining_life` получает свои числа

`remaining_life` считает условную кривую $S(a+u)/S(a)$ на собственной сетке из `n_grid = 1001` точки на $[0, \mathrm{horizon}]$ и снимает с неё четыре числа:

- `rul_q10`: первое $u$, при котором кривая не выше 0.9, то есть вероятность отказа дошла до 10%;
- `rul_median`: первое $u$, при котором кривая не выше 0.5;
- `rmrl`: площадь под кривой на $[0, \mathrm{horizon}]$ по правилу трапеций, среднее остаточное время, обрезанное горизонтом;
- `p_fail_window`: $1 - S(a+w)/S(a)$, посчитанная точно в $w$ = `window`; от сетки она не зависит.

«Первое» ищется булевой маской `curve <= level` с шейпом `(n, 1001)` и `argmax` вдоль оси времени, который возвращает индекс первого `True`. Если уровень в пределах горизонта не достигается, результат равен `inf`. Разрешение `rul_q10` и `rul_median` — один шаг сетки, `horizon / 1000`.

Числа дашборда даёт боевая модель, названная в `forecast_name` в `meta.json`; это не обязательно модель AFT основного прохода, поэтому раздел берёт `MODELS[SETTINGS.forecast]`.

Для тестовых дронов в эксплуатации `predictions.parquet` хранит те же четыре числа; ячейка сверяет их, если прослеживаемый дрон — один из них.

In [34]:
FORECAST_MODEL = MODELS[SETTINGS.forecast]
life = remaining_life(FORECAST_MODEL, drone, SETTINGS.horizon, SETTINGS.window)
peek("14.life", life)

N_LIFE_GRID = 1001
life_grid = np.linspace(0.0, SETTINGS.horizon, N_LIFE_GRID)
life_curve = FORECAST_MODEL.conditional_survival(drone, ages, life_grid)
reached_q10 = life_curve <= 0.9
reached_median = life_curve <= 0.5
first_q10 = reached_q10.argmax(axis=1)
first_median = reached_median.argmax(axis=1)
life_hand = {
    "rul_q10": np.where(reached_q10.any(axis=1), life_grid[first_q10], np.inf),
    "rul_median": np.where(reached_median.any(axis=1), life_grid[first_median], np.inf),
    "rmrl": np.trapezoid(life_curve, life_grid, axis=1),
    "p_fail_window": 1.0 - FORECAST_MODEL.conditional_survival(drone, ages, window_arr)[:, 0],
}
peek("14.life_grid", life_grid)
peek("14.life_curve", life_curve)
peek("14.reached_q10", reached_q10)
peek("14.reached_median", reached_median)
peek("14.first_q10", first_q10)
peek("14.first_median", first_median)
for key, value in life_hand.items():
    peek(f"14.life_hand.{key}", value)
    same(f"{key} = remaining_life", value, life[key])

if IN_TEST and bool(pred["in_service"].iloc[DRONE_POS]):
    for key in life_hand:
        same(f"сохранённый {key} = remaining_life", pred[key].iloc[[DRONE_POS]], life[key], rtol=1e-9)
else:
    print(f"{DRONE_ID} — не тестовый дрон в эксплуатации: в predictions.parquet остаточного ресурса для него нет")

print(
    f"\nшаг сетки {life_grid[1] - life_grid[0]:.4f} ч; "
    f"P(отказ за {window:g} ч) = {life_hand['p_fail_window'][0]:.4f}; "
    f"квантиль 10% остаточного ресурса = {life_hand['rul_q10'][0]:.4f} ч; "
    f"медиана = {life_hand['rul_median'][0]:.4f} ч; "
    f"среднее, обрезанное горизонтом, = {life_hand['rmrl'][0]:.2f} ч"
)
life

[14.life] DataFrame shape=(1, 6) dtype=5 x float64, 1 x str size=6 bytes=189
    столбцы: drone_id, age_h, rul_q10, rul_median, rmrl, p_fail_window
[14.life_grid] ndarray shape=(1001,) dtype=float64 size=1001 bytes=8008 | min=0 max=1111 nan=0 inf=0
    [   0.       1.111    2.222 ... 1108.778 1109.889 1111.   ]
[14.life_curve] ndarray shape=(1, 1001) dtype=float64 size=1001 bytes=8008 | min=0.00842644 max=1 nan=0 inf=0
    [[1.       0.994343 0.987299 ... 0.008426 0.008426 0.008426]]
[14.reached_q10] ndarray shape=(1, 1001) dtype=bool size=1001 bytes=1001 | true=985
    [[False False False ...  True  True  True]]
[14.reached_median] ndarray shape=(1, 1001) dtype=bool size=1001 bytes=1001 | true=904
    [[False False False ...  True  True  True]]
[14.first_q10] ndarray shape=(1,) dtype=int64 size=1 bytes=8 | min=16 max=16
    [16]
[14.first_median] ndarray shape=(1,) dtype=int64 size=1 bytes=8 | min=97 max=97
    [97]
[14.life_hand.rul_q10] ndarray shape=(1,) dtype=float64 size=1 bytes=

,drone_id,age_h,rul_q10,rul_median,rmrl,p_fail_window
0,DR-003057,193.5,17.776,107.767,178.111,0.47041


### 14.2 Условная кривая на дашборде

`figures.conditional_curves` рисует $S(a+u)/S(a)$ по `rul_grid`: 300 точек от 0 до `rul_horizon`. Ось x — **дополнительные** лётные часы $u$, а не общий налёт. Горизонталь на 0.5 отмечает уровень медианы; это фигура разметки, а не оценка по данным. Её x идёт от 0 до 1 в долях оси (`x domain`).

Карточка дрона рисует эту кривую под именем следа «Current conditions». Строку она пропускает через `common.whatif_base`, которая для дрона, уже выведенного из эксплуатации, начинает возраст с 0; эта ячейка всегда берёт `AGE`.

На графике 300 узлов, а у `remaining_life` 1001, поэтому медиана, снятая с графика, может отличаться от `rul_median` не больше чем на один шаг более грубой сетки.

In [35]:
cond_name = "Current conditions"
cond_title = f"Remaining life from {AGE:.0f} flight hours ({MODEL_LABELS[SETTINGS.forecast]})"
cond_fig = conditional_curves(FORECAST_MODEL, drone, ages, SETTINGS.rul_grid, [cond_name], cond_title)
describe_figure("14.cond_fig", cond_fig)

chart_curve = FORECAST_MODEL.conditional_survival(drone, ages, SETTINGS.rul_grid)
peek("14.chart_curve", chart_curve)
drawn = trace_xy(cond_fig, cond_name)
same("x следа = rul_grid", drawn["x"], SETTINGS.rul_grid)
same("y следа = conditional_survival(rul_grid)[0]", drawn["y"], chart_curve[0])
same("кривая в u = 0", chart_curve[:, :1], np.ones((len(drone), 1)))

cond_hline = cond_fig.layout.shapes[0]
same(
    "горизонталь на 0.5",
    np.array([cond_hline.y0, cond_hline.y1], dtype=float),
    np.array([0.5, 0.5]),
)

chart_reached = chart_curve[0] <= 0.5
median_chart = (
    float(SETTINGS.rul_grid[int(chart_reached.argmax())]) if chart_reached.any() else np.inf
)
rul_step = float(SETTINGS.rul_grid[1] - SETTINGS.rul_grid[0])
print(
    f"\nмедиана с графика (300 узлов) = {median_chart:.4f} ч; "
    f"remaining_life (1001 узел) = {life_hand['rul_median'][0]:.4f} ч; "
    f"шаг сетки графика {rul_step:.4f} ч"
)
display(cond_fig)

[14.cond_fig] Figure: следов 1, shapes 1, annotations 1
    заголовок: 'Remaining life from 194 flight hours (Cox)'
    ось x: 'Additional flight hours', range=None, rangemode=None
    ось y: 'S(a + u) / S(a)', range=(0, 1.02), rangemode=None
    высота: 400


,след,имя,тип,режим,шейп x,шейп y,x min,x max,y min,y max,NaN в y,цвет линии,толщина линии,заливка,в легенде
0,0,Current conditions,scatter,lines,"(300,)","(300,)",0,1111,0.00842644,1,0,#1f77b4,None,None,True


,вид,xref,x0,x1,yref,y0,y1,цвет,штрих,текст
0,shape/line,x domain,0,1,y,0.5,0.5,grey,dot,NaN
1,annotation,x domain,1,1,y,0.5,0.5,NaN,NaN,median


[14.chart_curve] ndarray shape=(1, 300) dtype=float64 size=300 bytes=2400 | min=0.00842644 max=1 nan=0 inf=0
    [[1.       0.973341 0.948516 ... 0.008616 0.008616 0.008426]]
[ok] x следа = rul_grid: shape=(300,), max |разница| = 0.000e+00
[ok] y следа = conditional_survival(rul_grid)[0]: shape=(300,), max |разница| = 0.000e+00
[ok] кривая в u = 0: shape=(1, 1), max |разница| = 0.000e+00
[ok] горизонталь на 0.5: shape=(2,), max |разница| = 0.000e+00

медиана с графика (300 узлов) = 107.7559 ч; remaining_life (1001 узел) = 107.7670 ч; шаг сетки графика 3.7157 ч


## 15. От одного дрона к кривой интенсивности флота

`figures.hazard` рисует не строку одного дрона. Она рисует `hazard_table`: три кривые на `hazard_grid`, посчитанные по всем тестовым дронам, — наблюдаемую интенсивность с её интервалом и по одной предсказанной групповой интенсивности на модель. График — проверка качества на тестовой выборке, поэтому две кривые моделей на нём построены по **оценочным** моделям (`EVAL_MODELS`), а не по боевым.

### 15.1 Наблюдаемая интенсивность: сглаженная оценка Нельсона–Аалена

`observed_hazard` использует только `duration_h` и `event`, без модели:

1. различные моменты отказов $t_j$ и число отказов $d_j$ в каждом, шейп `(k,)`;
2. число дронов, ещё находящихся под наблюдением непосредственно перед каждым моментом отказа, $n_j$, шейп `(k,)`;
3. приращения Нельсона–Аалена $d_j / n_j$ — доля дронов под риском, отказавших в $t_j$;
4. веса Епанечникова между каждой точкой сетки и каждым моментом отказа, шейп `(g, k)`;
5. сглаженная интенсивность $h(t) = \sum_j K_b(t - t_j)\, d_j / n_j$, шейп `(g,)`, и её стандартная ошибка $\sqrt{\sum_j K_b(t - t_j)^2\, d_j / n_j^2}$;
6. поточечный 95% интервал $h \pm 1.96\,\mathrm{se}$, нижняя граница обрезается нулём.

In [36]:
fleet_grid = SETTINGS.hazard_grid
bandwidth = SETTINGS.hazard_bandwidth
durations = pred["duration_h"].to_numpy(dtype=float)
events = pred["event"].to_numpy(dtype=bool)

obs_times, obs_deaths = np.unique(durations[events], return_counts=True)
obs_at_risk = len(durations) - np.searchsorted(np.sort(durations), obs_times, side="left")
obs_increments = obs_deaths / obs_at_risk
obs_weights = _epanechnikov(fleet_grid, obs_times, bandwidth)
obs_h_hand = obs_weights @ obs_increments
obs_se = np.sqrt(obs_weights**2 @ (obs_deaths / obs_at_risk**2))
obs_low_hand = np.maximum(obs_h_hand - 1.96 * obs_se, 0.0)
obs_high_hand = obs_h_hand + 1.96 * obs_se
obs_h, obs_low, obs_high = observed_hazard(durations, events, fleet_grid, bandwidth)

peek("15.durations", durations)
peek("15.events", events)
peek("15.obs_times", obs_times)
peek("15.obs_deaths", obs_deaths)
peek("15.obs_at_risk", obs_at_risk)
peek("15.obs_increments", obs_increments)
peek("15.obs_weights", obs_weights)
peek("15.obs_se", obs_se)
peek("15.obs_h", obs_h)
peek("15.obs_low", obs_low)
peek("15.obs_high", obs_high)
same("наблюдаемая интенсивность", obs_h, obs_h_hand)
same("нижняя граница", obs_low, obs_low_hand)
same("верхняя граница", obs_high, obs_high_hand)

print("\nпервые моменты отказов:")
pd.DataFrame(
    {
        "момент отказа": obs_times,
        "отказов d": obs_deaths,
        "под риском n": obs_at_risk,
        "приращение d/n": obs_increments,
    }
).head()

[15.durations] ndarray shape=(1001,) dtype=float64 size=1001 bytes=8008 | min=0.18 max=1236.24 nan=0 inf=0
    [ 63.67 159.67 312.4  ...  66.78 227.58   1.03]
[15.events] ndarray shape=(1001,) dtype=bool size=1001 bytes=1001 | true=448
    [ True  True False ...  True False  True]
[15.obs_times] ndarray shape=(442,) dtype=float64 size=442 bytes=3536 | min=0.55 max=1156.68 nan=0 inf=0
    [5.50000e-01 9.20000e-01 9.60000e-01 ... 7.60620e+02 8.05900e+02 1.15668e+03]
[15.obs_deaths] ndarray shape=(442,) dtype=int64 size=442 bytes=3536 | min=1 max=2
    [1 1 1 ... 1 1 1]
[15.obs_at_risk] ndarray shape=(442,) dtype=int64 size=442 bytes=3536 | min=3 max=1000
    [1000  999  998 ...   25   20    3]
[15.obs_increments] ndarray shape=(442,) dtype=float64 size=442 bytes=3536 | min=0.001 max=0.333333 nan=0 inf=0
    [0.001    0.001001 0.001002 ... 0.04     0.05     0.333333]
[15.obs_weights] ndarray shape=(200, 442) dtype=float64 size=88400 bytes=707200 | min=0 max=0.0176723 nan=0 inf=0
    [[0.0

,момент отказа,отказов d,под риском n,приращение d/n
0,0.55,1,1000,0.001
1,0.92,1,999,0.001001
2,0.96,1,998,0.001002
3,1.03,2,997,0.00200602
4,1.2,1,995,0.00100503


### 15.2 Предсказанная групповая интенсивность: среднее, взвешенное выживаемостью

`population_hazard` считает модель на каждом тестовом дроне и в каждой точке сетки: `survival` и `hazard` обе возвращают `(n, g)`. Дроны, которые по модели к моменту $t$ уже отказали бы, должны выйти из среднего, как отказавшие дроны выходят из наблюдаемого множества риска, поэтому каждый дрон взвешивается своей вероятностью выживания:

$$h_{\mathrm{group}}(t) = \sum_i S_i(t)\, h_i(t) \,/\, \sum_i S_i(t).$$

Суммы берутся по оси дронов (`axis=0`) и оставляют одно значение на точку сетки, `(g,)`. Простое среднее $h_i$ на график не попадает; ячейка хранит его только затем, чтобы показать разницу.

Если прослеживаемый дрон есть в тестовой таблице, его строка матрицы интенсивностей `(n, g)` — одно слагаемое суммы, и она равна интенсивности одного дрона, посчитанной на этой сетке.

In [37]:
FLEET_SURV, FLEET_HAZARD = {}, {}
WEIGHTED_SUM, WEIGHT_TOTAL = {}, {}
GROUP_HAZARD, PLAIN_MEAN_HAZARD = {}, {}
for kind, model in EVAL_MODELS.items():
    FLEET_SURV[kind] = model.survival(pred, fleet_grid)
    FLEET_HAZARD[kind] = model.hazard(pred, fleet_grid, bandwidth)
    WEIGHTED_SUM[kind] = (FLEET_SURV[kind] * FLEET_HAZARD[kind]).sum(axis=0)
    WEIGHT_TOTAL[kind] = FLEET_SURV[kind].sum(axis=0)
    GROUP_HAZARD[kind] = WEIGHTED_SUM[kind] / WEIGHT_TOTAL[kind]
    PLAIN_MEAN_HAZARD[kind] = FLEET_HAZARD[kind].mean(axis=0)
    peek(f"15.FLEET_SURV.{kind}", FLEET_SURV[kind])
    peek(f"15.FLEET_HAZARD.{kind}", FLEET_HAZARD[kind])
    peek(f"15.WEIGHTED_SUM.{kind}", WEIGHTED_SUM[kind])
    peek(f"15.WEIGHT_TOTAL.{kind}", WEIGHT_TOTAL[kind])
    peek(f"15.GROUP_HAZARD.{kind}", GROUP_HAZARD[kind])
    peek(f"15.PLAIN_MEAN_HAZARD.{kind}", PLAIN_MEAN_HAZARD[kind])
    same(
        f"{kind}: sum(S h) / sum(S) = population_hazard",
        GROUP_HAZARD[kind],
        population_hazard(model, pred, fleet_grid, bandwidth),
    )
    if IN_TEST:
        same(
            f"{kind}: строка {DRONE_POS} матрицы интенсивностей = hazard(drone)",
            FLEET_HAZARD[kind][DRONE_POS],
            model.hazard(drone, fleet_grid, bandwidth)[0],
        )

[15.FLEET_SURV.cox] ndarray shape=(1001, 200) dtype=float64 size=200200 bytes=1601600 | min=2.13113e-06 max=0.983396 nan=0 inf=0
    [[0.968453 0.966704 0.965286 ... 0.650272 0.64988  0.649095]
     [0.966053 0.964175 0.962652 ... 0.630216 0.62981  0.628998]
     [0.945451 0.942477 0.940069 ... 0.485063 0.484591 0.483648]
     ...
     [0.93627  0.932819 0.930026 ... 0.433331 0.432852 0.431893]
     [0.977226 0.975956 0.974926 ... 0.730363 0.730037 0.729382]
     [0.822317 0.813519 0.806457 ... 0.124161 0.123872 0.123295]]
[15.FLEET_HAZARD.cox] ndarray shape=(1001, 200) dtype=float64 size=200200 bytes=1601600 | min=0.000350478 max=0.055364 nan=0 inf=0
    [[0.000682 0.000677 0.000673 ... 0.000864 0.000865 0.00087 ]
     [0.000735 0.000729 0.000725 ... 0.000923 0.000924 0.000929]
     [0.001189 0.00118  0.001172 ... 0.001394 0.001395 0.001403]
     ...
     [0.001394 0.001383 0.001374 ... 0.001586 0.001587 0.001596]
     [0.000491 0.000487 0.000484 ... 0.000641 0.000642 0.000646]
     [

Одна точка сетки делает взвешивание наглядным. Ячейка берёт узел `hazard_grid`, ближайший к возрасту дрона, и показывает для каждой модели взвешенное и простое среднее, суммарный вес $\sum_i S_i$ и, если дрон есть в тестовой таблице, его долю этого веса и его собственную интенсивность.

In [38]:
point_i, t_star = nearest(fleet_grid)
point_rows = []
for kind in EVAL_MODELS:
    surv_col = FLEET_SURV[kind][:, point_i]
    rate_col = FLEET_HAZARD[kind][:, point_i]
    point_row = {
        "источник": kind,
        "h группы, на час": GROUP_HAZARD[kind][point_i],
        f"h группы, на {HAZARD_SCALE:g} ч": HAZARD_SCALE * GROUP_HAZARD[kind][point_i],
        "простое среднее h, на час": PLAIN_MEAN_HAZARD[kind][point_i],
        "сумма S": WEIGHT_TOTAL[kind][point_i],
    }
    if IN_TEST:
        point_row["доля веса дрона"] = surv_col[DRONE_POS] / surv_col.sum()
        point_row["h дрона, на час"] = rate_col[DRONE_POS]
    point_rows.append(point_row)

point_table = pd.DataFrame(point_rows)
peek("15.point_table", point_table)
print(
    f"\nузел hazard_grid рядом с возрастом: {t_star:.2f} ч (индекс {point_i}); "
    f"наблюдаемая скорость {HAZARD_SCALE * obs_h[point_i]:.4f} на {HAZARD_SCALE:g} ч, "
    f"95% интервал [{HAZARD_SCALE * obs_low[point_i]:.4f}, {HAZARD_SCALE * obs_high[point_i]:.4f}]"
)
point_table

[15.point_table] DataFrame shape=(2, 7) dtype=6 x float64, 1 x str size=14 bytes=250
    столбцы: источник, h группы, на час, h группы, на 100 ч, простое среднее h, на час, сумма S, доля
    веса дрона, h дрона, на час

узел hazard_grid рядом с возрастом: 192.77 ч (индекс 66); наблюдаемая скорость 0.1646 на 100 ч, 95% интервал [0.1248, 0.2045]


,источник,"h группы, на час","h группы, на 100 ч","простое среднее h, на час",сумма S,доля веса дрона,"h дрона, на час"
0,cox,0.00156395,0.156395,0.00202334,716.426,0.000362402,0.00636831
1,aft,0.00178027,0.178027,0.00221337,685.724,0.000365319,0.00526702


### 15.3 Таблица и график

`hazard_table` складывает три кривые в длинный формат: `g` строк на источник (`observed`, `cox`, `aft`) со столбцами `t` и `h`, на лётный час. `ci_low` и `ci_high` заполнены только у строк `observed`.

`figures.hazard` умножает всё на `HAZARD_SCALE = 100`, поэтому ось y — отказы на 100 лётных часов. Со списком моделей (`kinds`) она добавляет пять следов в таком порядке:

1. верхняя граница интервала, невидимая линия толщины 0;
2. нижняя граница с заливкой до предыдущего следа (`fill="tonexty"`); эта пара даёт серую полосу;
3. наблюдаемая линия;
4. и 5. по одной линии на модель.

У двух следов полосы нет имени, и в легенде они скрыты. Серая полоса — 95% интервал **наблюдаемой** интенсивности, а не разброс между моделями. Дашборд рисует этот график и только для выбранной модели, тогда следов четыре.

In [39]:
table = hazard_table(DATA, SETTINGS)
peek("15.hazard_table", table)
print(f"строк на источник: {table['source'].value_counts().to_dict()}")

observed = table[table["source"] == "observed"]
same("таблица: observed h", observed["h"], obs_h)
same("таблица: ci_low", observed["ci_low"], obs_low)
same("таблица: ci_high", observed["ci_high"], obs_high)
for kind in EVAL_MODELS:
    same(f"таблица: h для {kind} = групповая интенсивность", table.loc[table["source"] == kind, "h"], GROUP_HAZARD[kind])

fleet_fig = fleet_hazard(table, list(EVAL_MODELS))
describe_figure("15.fleet_fig", fleet_fig)

band = [i for i, tr in enumerate(fleet_fig.data) if not tr.name]
if len(band) != 2:
    raise RuntimeError(f"у полосы {len(band)} безымянных следов, ожидались два")
same(
    f"верхний край полосы = {HAZARD_SCALE:g} * ci_high",
    trace_xy(fleet_fig, band[0])["y"],
    HAZARD_SCALE * obs_high,
)
same(
    f"нижний край полосы = {HAZARD_SCALE:g} * ci_low",
    trace_xy(fleet_fig, band[1])["y"],
    HAZARD_SCALE * obs_low,
)
same(
    f"наблюдаемая линия = {HAZARD_SCALE:g} * h",
    trace_xy(fleet_fig, 2)["y"],
    HAZARD_SCALE * obs_h,
)
for kind in EVAL_MODELS:
    drawn = trace_xy(fleet_fig, f"{MODEL_LABELS[kind]}: prediction")
    same(f"линия {kind}: x = hazard_grid", drawn["x"], fleet_grid)
    same(f"линия {kind}: y = {HAZARD_SCALE:g} * групповая интенсивность", drawn["y"], HAZARD_SCALE * GROUP_HAZARD[kind])

display(fleet_fig)
print("первые строки каждого источника:")
table.groupby("source", sort=False).head(2)

[15.hazard_table] DataFrame shape=(600, 5) dtype=4 x float64, 1 x str size=3000 bytes=27007
    столбцы: source, t, h, ci_low, ci_high
строк на источник: {'observed': 200, 'cox': 200, 'aft': 200}
[ok] таблица: observed h: shape=(200,), max |разница| = 0.000e+00
[ok] таблица: ci_low: shape=(200,), max |разница| = 0.000e+00
[ok] таблица: ci_high: shape=(200,), max |разница| = 0.000e+00
[ok] таблица: h для cox = групповая интенсивность: shape=(200,), max |разница| = 0.000e+00
[ok] таблица: h для aft = групповая интенсивность: shape=(200,), max |разница| = 0.000e+00
[15.fleet_fig] Figure: следов 5, shapes 0, annotations 0
    заголовок: 'Failure rate: predicted vs observed'
    ось x: 'Flight hours', range=None, rangemode=None
    ось y: 'Failures per 100 flight hours', range=None, rangemode=tozero
    высота: 400


,след,имя,тип,режим,шейп x,шейп y,x min,x max,y min,y max,NaN в y,цвет линии,толщина линии,заливка,в легенде
0,0,NaN,scatter,lines,"(200,)","(200,)",42.4393,495.71,0.182687,0.309557,0,NaN,0,NaN,False
1,1,NaN,scatter,lines,"(200,)","(200,)",42.4393,495.71,0.0383456,0.16041,0,NaN,0,tonexty,False
2,2,Kaplan-Meier (observed),scatter,lines,"(200,)","(200,)",42.4393,495.71,0.121227,0.228797,0,#222222,NaN,NaN,True
3,3,Cox: prediction,scatter,lines,"(200,)","(200,)",42.4393,495.71,0.129324,0.182534,0,#1f77b4,NaN,NaN,True
4,4,AFT: prediction,scatter,lines,"(200,)","(200,)",42.4393,495.71,0.12594,0.212727,0,#ff7f0e,NaN,NaN,True


[ok] верхний край полосы = 100 * ci_high: shape=(200,), max |разница| = 0.000e+00
[ok] нижний край полосы = 100 * ci_low: shape=(200,), max |разница| = 0.000e+00
[ok] наблюдаемая линия = 100 * h: shape=(200,), max |разница| = 0.000e+00
[ok] линия cox: x = hazard_grid: shape=(200,), max |разница| = 0.000e+00
[ok] линия cox: y = 100 * групповая интенсивность: shape=(200,), max |разница| = 0.000e+00
[ok] линия aft: x = hazard_grid: shape=(200,), max |разница| = 0.000e+00
[ok] линия aft: y = 100 * групповая интенсивность: shape=(200,), max |разница| = 0.000e+00


первые строки каждого источника:


,source,t,h,ci_low,ci_high
0,observed,42.4393,0.00193579,0.00159572,0.00227586
1,observed,44.717,0.00190809,0.00156975,0.00224642
200,cox,42.4393,0.00176073,NaN,NaN
201,cox,44.717,0.00173427,NaN,NaN
400,aft,42.4393,0.00212408,NaN,NaN
401,aft,44.717,0.002126,NaN,NaN


## 16. Индекс сохранённых промежуточных результатов

Каждый объект, переданный в `peek` или `describe_figure`, лежит в `TRACE`. Таблица перечисляет их в порядке прохода: тип, шейп, dtype, число элементов и память в байтах.

Чтобы рассмотреть любой из них, обратитесь к словарю в новой ячейке, например `TRACE["4.z"]`, `TRACE["9.kernel"].shape` или `TRACE["15.fleet_fig"].data[2].y`. Переменные верхнего уровня из ячеек выше тоже доступны под своими именами.

In [40]:
captured = trace_index()
print(f"сохранено объектов: {len(captured)}")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 60):
    display(captured)

сохранено объектов: 150


,name,type,shape,dtype,size,bytes
0,1.META,dict,None,NaN,21,<NA>
1,1.MODEL.params,dict,None,NaN,8,<NA>
2,1.MODEL.features,list,None,NaN,16,<NA>
3,1.MODEL.cat_features,list,None,NaN,4,<NA>
4,1.MODEL.categories,dict,None,NaN,3,<NA>
5,1.MODEL.calibration,NoneType,None,NaN,<NA>,<NA>
6,1.MODEL._breslow,NoneType,None,NaN,<NA>,<NA>
7,1.MODEL.history_lengths,dict,None,NaN,1,<NA>
8,1.fleet,DataFrame,"(5000, 21)","11 x float64, 5 x str, 4 x int64, 1 x bool",105000,1021733
9,1.drone,DataFrame,"(1, 21)","11 x float64, 5 x str, 4 x int64, 1 x bool",21,340


### Куда смотреть в первую очередь

- **Margin отличается от вашего собственного `booster.predict`**: раздел 2.3 (`output_margin`, `iteration_range`) и раздел 2.2 (порядок и типы признаков).
- **Неожиданное предсказание для нового дрона**: раздел 2.1 (неизвестная категория становится NaN) и раздел 2.5 (какие признаки сдвигают margin).
- **Значение в этом ноутбуке и на дашборде разное**: разделы 11–13 (кривая считается в узлах сетки, а не в точном возрасте, а интенсивность Cox зависит от ширины ядра).
- **Кривая Cox выходит на плато**: раздел 9.5 (базовая линия заканчивается на последнем отказе обучения).
- **Сохранённые предсказания или метрики перестали сходиться**: раздел 10 (артефакты и файлы моделей получены в разных прогонах).